# Modelo Anual — OLS-Augmented (v8c + OLS_Pred feature)
## Ensemble Segmentado con OLS como Variable: SmallMed (LGB+RF+SARIMAX) + LargeColossal (LGB)

**Diferencia vs Modelo_Anual_FinalV1**: se agrega `OLS_Pred` como feature adicional en los modelos
multi-horizonte. El OLS se pre-computa por rolling origin (sin fuga de datos, igual que SARIMAX)
y provee la prediccion lineal como referencia. Los modelos ML aprenden cuando corregir el OLS.

**Nuevos experimentos**: `LGB_MultiH_OLS`, `RF_MultiH_OLS`, `LGB_MultiH_LC_OLS`, `RF_MultiH_LC_OLS`


## 1. Imports y Configuración

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, copy

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.model_selection import TimeSeriesSplit

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings('ignore')

HORIZONS        = list(range(1, 8))
H_FOCUS         = [5, 6, 7]
BASE_YEAR       = 2025
ROLLING_ORIGINS = [2010,2011,2012,2013,2014,2015,2016,2017,2018]
LGB_TRIALS      = 50
XGB_TRIALS      = 20
CB_TRIALS       = 20
RF_TRIALS       = 20
TUNE_H          = 5
MAX_LEAVES      = 20
EXPORT_DIR      = 'outputs_best'
os.makedirs(EXPORT_DIR, exist_ok=True)

EXCLUDE_MINES  = {'spence', 'quebrada blanca'}

# COVID-19 pandemic: exogenous shock indicator
# NOTE: always 0 in training (origins 2010-2018 → max target = 2025 but training requires
#       Target_Year <= origin_year <= 2018). Is_Pandemic_Target is 1 only for validation
#       test years 2020-2021. Set to 0 for 2026-2032 projections.
PANDEMIC_YEARS = {2020, 2021}

print('Configuración cargada')
print(f'Orígenes: {ROLLING_ORIGINS}')
print(f'Horizontes foco: H+{H_FOCUS}')
print(f'Pandemic years (exog. flag): {sorted(PANDEMIC_YEARS)}')

Configuración cargada
Orígenes: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]
Horizontes foco: H+[5, 6, 7]
Pandemic years (exog. flag): [2020, 2021]


## 2. Wrappers de Modelos y Optuna

In [2]:
class LGBWrapper(BaseEstimator, RegressorMixin):
    _estimator_type = 'regressor'
    def __init__(self, n_estimators=500, learning_rate=0.03, num_leaves=15,
                 min_child_samples=7, reg_alpha=0.1, reg_lambda=1.0):
        self.n_estimators=n_estimators; self.learning_rate=learning_rate
        self.num_leaves=num_leaves; self.min_child_samples=min_child_samples
        self.reg_alpha=reg_alpha; self.reg_lambda=reg_lambda
    def fit(self, X, y, **kw):
        self.model_ = lgb.LGBMRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            num_leaves=self.num_leaves, min_child_samples=self.min_child_samples,
            reg_alpha=self.reg_alpha, reg_lambda=self.reg_lambda,
            random_state=42, verbose=-1)
        self.model_.fit(X, y); return self
    def predict(self, X):
        check_is_fitted(self,'model_'); return self.model_.predict(X)

class XGBWrapper(BaseEstimator, RegressorMixin):
    _estimator_type = 'regressor'
    def __init__(self, n_estimators=400, learning_rate=0.03, max_depth=4,
                 min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0, subsample=0.8):
        self.n_estimators=n_estimators; self.learning_rate=learning_rate
        self.max_depth=max_depth; self.min_child_weight=min_child_weight
        self.reg_alpha=reg_alpha; self.reg_lambda=reg_lambda; self.subsample=subsample
    def fit(self, X, y, **kw):
        self.model_ = xgb.XGBRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            max_depth=self.max_depth, min_child_weight=self.min_child_weight,
            reg_alpha=self.reg_alpha, reg_lambda=self.reg_lambda,
            subsample=self.subsample, random_state=42, verbosity=0)
        self.model_.fit(X, y); return self
    def predict(self, X):
        check_is_fitted(self,'model_'); return self.model_.predict(X)

class CBWrapper(BaseEstimator, RegressorMixin):
    _estimator_type = 'regressor'
    def __init__(self, iterations=400, learning_rate=0.03, depth=4,
                 l2_leaf_reg=3.0, min_data_in_leaf=5):
        self.iterations=iterations; self.learning_rate=learning_rate
        self.depth=depth; self.l2_leaf_reg=l2_leaf_reg
        self.min_data_in_leaf=min_data_in_leaf
    def fit(self, X, y, cat_features=None, **kw):
        self.model_ = CatBoostRegressor(
            iterations=self.iterations, learning_rate=self.learning_rate,
            depth=self.depth, l2_leaf_reg=self.l2_leaf_reg,
            min_data_in_leaf=self.min_data_in_leaf,
            random_seed=42, verbose=False)
        self.model_.fit(X, y, cat_features=cat_features or []); return self
    def predict(self, X):
        check_is_fitted(self,'model_'); return self.model_.predict(X)

class RFWrapper(BaseEstimator, RegressorMixin):
    _estimator_type = 'regressor'
    def __init__(self, n_estimators=200, max_depth=8, min_samples_leaf=5, max_features='sqrt'):
        self.n_estimators=n_estimators; self.max_depth=max_depth
        self.min_samples_leaf=min_samples_leaf; self.max_features=max_features
    def fit(self, X, y, **kw):
        self.model_ = RandomForestRegressor(
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf, max_features=self.max_features,
            random_state=42, n_jobs=-1)
        self.model_.fit(X, y); return self
    def predict(self, X):
        check_is_fitted(self, 'model_'); return self.model_.predict(X)

class SARIMAXPerMine:
    def __init__(self):
        self.fitted_={}
    def fit(self, df):
        self.fitted_={}
        for mine in df['Match_Key'].unique():
            sub  = df[df['Match_Key']==mine].sort_values('Anio')
            prod = sub['Produccion'].values.astype(float)
            prod = prod[np.argmax(prod>0):]
            if len(prod)<6: self.fitted_[mine]=None; continue
            for ord_ in [(1,1,1),(1,1,0),(0,1,1),(0,1,0)]:
                try: self.fitted_[mine]=ARIMA(prod,order=ord_).fit(); break
                except: continue
            else: self.fitted_[mine]=None
        return self
    def predict_h(self, mine, h):
        res=self.fitted_.get(mine)
        if res is None: return np.nan
        try:
            fc=res.forecast(h)
            return max(0.0, float(fc.iloc[-1] if hasattr(fc,'iloc') else fc[-1]))
        except: return np.nan

def make_optuna_tuner(algo, n_trials, max_leaves=MAX_LEAVES):
    tscv = TimeSeriesSplit(n_splits=3)
    def tune(X, y):
        def obj(trial):
            if algo == 'lgb':
                params = dict(
                    n_estimators      = trial.suggest_int('n', 200, 800),
                    learning_rate     = trial.suggest_float('lr', 0.005, 0.10, log=True),
                    num_leaves        = trial.suggest_int('nl', 8, max_leaves),
                    min_child_samples = trial.suggest_int('mcs', 5, 25),
                    reg_alpha         = trial.suggest_float('ra', 0.0, 2.0),
                    reg_lambda        = trial.suggest_float('rl', 0.5, 3.0))
                m = LGBWrapper(**params)
            elif algo == 'xgb':
                params = dict(
                    n_estimators     = trial.suggest_int('n', 100, 600),
                    learning_rate    = trial.suggest_float('lr', 0.005, 0.15, log=True),
                    max_depth        = trial.suggest_int('md', 3, 6),
                    min_child_weight = trial.suggest_int('mcw', 3, 20),
                    reg_alpha        = trial.suggest_float('ra', 0.0, 2.0),
                    reg_lambda       = trial.suggest_float('rl', 0.5, 3.0),
                    subsample        = trial.suggest_float('ss', 0.6, 1.0))
                m = XGBWrapper(**params)
            elif algo == 'rf':
                params = dict(
                    n_estimators     = trial.suggest_int('n', 100, 400),
                    max_depth        = trial.suggest_int('md', 4, 12),
                    min_samples_leaf = trial.suggest_int('msl', 3, 20),
                    max_features     = trial.suggest_categorical('mf', ['sqrt', 'log2', 0.5]))
                m = RFWrapper(**params)
            else:
                params = dict(
                    iterations       = trial.suggest_int('n', 100, 500),
                    learning_rate    = trial.suggest_float('lr', 0.01, 0.15, log=True),
                    depth            = trial.suggest_int('d', 3, 6),
                    l2_leaf_reg      = trial.suggest_float('l2', 1.0, 5.0),
                    min_data_in_leaf = trial.suggest_int('mdl', 3, 15))
                m = CBWrapper(**params)
            maes = []
            for ti, vi in tscv.split(X):
                if len(X[ti]) < 5: continue
                try:
                    mm = copy.deepcopy(m); mm.fit(X[ti], y[ti])
                    maes.append(np.median(np.abs(y[vi] - mm.predict(X[vi]))))
                except: maes.append(1e9)
            return np.mean(maes) if maes else 1e9  # mean of per-fold MdAE
        st = optuna.create_study(direction='minimize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
        st.optimize(obj, n_trials=n_trials, show_progress_bar=False)
        rn = {'n':'n_estimators','lr':'learning_rate','nl':'num_leaves',
              'mcs':'min_child_samples','ra':'reg_alpha','rl':'reg_lambda',
              'md':'max_depth','mcw':'min_child_weight','ss':'subsample','msl':'min_samples_leaf','mf':'max_features',
              'd':'depth','l2':'l2_leaf_reg','mdl':'min_data_in_leaf'}
        return {rn.get(k,k):v for k,v in st.best_params.items()}
    return tune

print('Wrappers y Optuna definidos')

Wrappers y Optuna definidos


## 3. Carga de Datos y Feature Engineering

In [3]:
df_raw = pd.read_csv('../../../01_Data/processed/produccion_master.csv', encoding='utf-8')
df_raw['Match_Key']       = df_raw['Match_Key'].str.lower().str.strip()
df_raw['Produccion']      = df_raw['Produccion'].fillna(0)
df_raw['Inversion_MMUSD'] = df_raw['Inversion_MMUSD'].fillna(0)
df_raw['Capital_Stock']   = df_raw['Capital_Stock'].fillna(0)
df_raw['Precio_Cobre']    = df_raw['Precio_Cobre'].ffill().fillna(0)
df_raw = df_raw[~df_raw['Match_Key'].isin(EXCLUDE_MINES)].reset_index(drop=True)
df_raw = df_raw.sort_values(['Match_Key','Anio']).reset_index(drop=True)

COMPANY_SIZE_MAP = {
    'escondida':2,'chuquicamata':2,'el teniente':2,'andina':2,'radomiro tomic':2,
    'salvador':2,'ministro hales':2,'gabriela mistral':2,'collahuasi':2,
    'los bronces':2,'lomas bayas':2,'cerro colorado':2,'el abra':2,
    'los pelambres':1,'centinela_centinela_sulfuros_':1,'zaldivar':1,
    'antucoya':1,'michilla':1,'andacollo':1,'candelaria':1,
    'caserones':1,'sierra gorda':1,'centinela_centinela_óxidos_':1,
    'capstone copper (4)':0,
}
for k in df_raw['Match_Key'].unique():
    if k not in COMPANY_SIZE_MAP: COMPANY_SIZE_MAP[k]=1

MINES = sorted(df_raw['Match_Key'].unique())
print(f'Dataset: {len(df_raw):,} rows | {len(MINES)} minas')

SIZE_LBL = {0:'Small', 1:'Medium', 2:'Large', 3:'Colossal'}

def compute_mine_size(df_raw, origin_year):
    end, start = origin_year-1, origin_year-6
    avgs = {}
    for mine in MINES:
        s = df_raw[(df_raw['Match_Key']==mine) & df_raw['Anio'].between(start,end)]['Produccion']
        avgs[mine] = float(s.mean()) if (len(s)>0 and s.sum()>0) else 0.0
    vals = list(avgs.values())
    q25, q50, q75 = np.percentile(vals, 25), np.percentile(vals, 50), np.percentile(vals, 75)
    return {m: (0 if v<=q25 else (1 if v<=q50 else (2 if v<=q75 else 3))) for m, v in avgs.items()}

FIRST_PROD_YEAR = {}
for mine in MINES:
    sub=df_raw[(df_raw['Match_Key']==mine)&(df_raw['Produccion']>0)]['Anio']
    FIRST_PROD_YEAR[mine]=int(sub.min()) if len(sub)>0 else 1982

# Training max Mine_age (origins 2010-2018, oldest mines started 1982 → max age 36)
# Cap prevents OOD extrapolation at projection time (2025: oldest mines reach age 43)
MINE_AGE_CAP = 36

def crear_features(df_raw):
    df = df_raw.copy().sort_values(['Match_Key','Anio']).reset_index(drop=True)
    g  = lambda col: df.groupby('Match_Key')[col]
    for lag in [1,2,3,5]:
        df[f'Prod_Lag{lag}'] = g('Produccion').shift(lag)
    def _trend(s):
        if len(s)<2: return 0.0
        try: return float(np.polyfit(np.arange(len(s)),s,1)[0])
        except: return 0.0
    df['Tendencia_5y'] = g('Produccion').transform(
        lambda x: x.shift(1).rolling(5,min_periods=2).apply(_trend,raw=True))
    prod_lag1 = g('Produccion').shift(1)
    df['Prod_pct_change'] = ((df['Produccion']-prod_lag1)/(prod_lag1.abs()+1)).clip(-2,2)
    # Mine_age capped at MINE_AGE_CAP (36) — training max age is 36 (mines started 1982, origin 2018)
    _raw_age = df.apply(lambda r: max(0, r['Anio']-FIRST_PROD_YEAR.get(r['Match_Key'],1982)), axis=1)
    df['Mine_age'] = _raw_age.clip(upper=MINE_AGE_CAP)
    cu_lag = df.groupby('Match_Key')['Precio_Cobre'].shift(1)
    df['Cu_lag1'] = cu_lag
    df['Cu_regime'] = df.groupby('Match_Key')['Cu_lag1'].transform(
        lambda x: x.rolling(10, min_periods=3).rank(pct=True)).fillna(0.5)
    total_by_year = df.groupby('Anio')['Produccion'].transform('sum')
    df['Mine_share'] = (df['Prod_Lag1'] / (total_by_year.shift(1) + 1)).clip(0, 1)
    df['Capital_Stock_Lag1'] = df.groupby('Match_Key')['Capital_Stock'].shift(1).fillna(0)
    df['Company_Size'] = df['Match_Key'].map(COMPANY_SIZE_MAP).fillna(1).astype(int)
    df['Mine_Size']    = 0
    # Diagnostic columns (not in E6_BASE — excluded by ablation study)
    df['Prod_HistMax']    = g('Produccion').transform(lambda x: x.shift(1).expanding().max())
    df['Prod_vs_HistMax'] = (df['Prod_Lag1'] / (df['Prod_HistMax'] + 1e-6)).clip(0, 1)
    df['Is_Decline']      = ((df['Tendencia_5y'] < 0) & (df['Prod_vs_HistMax'] < 0.85)).astype(int)
    return df

df_feats = crear_features(df_raw)

# Is_Pandemic_Target is added dynamically inside the ML loop (depends on horizon h).
# For 2026-2032 projections: set Is_Pandemic_Target = 0.
#
# v8c SPLIT feature sets (ablation confirmed):
#   - SM keeps V7 baseline (9 feats) — Capital_Stock_Lag1 hurt SM when Optuna re-tuned
#   - LC uses V7 + Capital_Stock_Lag1 (10 feats) — confirmed +2.1pp WR for LC segment
#   - Prod_vs_HistMax + Is_Decline hurt H+1 by -3 to -4pp → removed from both
#   - SEIA_pipeline_log = all zeros (file not available) → removed
#   - Company_Size was accidentally missing in v8b → restored
E6_BASE_SM = ['Company_Size','Mine_Size','Prod_Lag1','Tendencia_5y',
              'Prod_pct_change','Mine_age','Mine_share',
              'Is_Pandemic_Target','Cu_regime']           # 9 features (V7 baseline)
E6_BASE_LC = E6_BASE_SM + ['Capital_Stock_Lag1']         # 10 features (V7 + CapStock)
E6_MULTI_SM = E6_BASE_SM + ['Horizonte_feat']            # 10 features
E6_MULTI_LC = E6_BASE_LC + ['Horizonte_feat']            # 11 features
# Aliases for backward-compat with diagnostic / SHAP cells
E6_BASE  = E6_BASE_SM
E6_MULTI = E6_MULTI_SM
# OLS-augmented feature sets: OLS_Pred = linear model prediction (pre-computed per origin)
# The gradient/RF models learn to correct the OLS linear baseline when needed
E6_MULTI_SM_OLS = E6_MULTI_SM + ['OLS_Pred']   # 11 features (SM + linear pred)
E6_MULTI_LC_OLS = E6_MULTI_LC + ['OLS_Pred']   # 12 features (LC + linear pred)


# Experiments — SmallMed uses V7, LargeColossal uses V7+Capital_Stock_Lag1
EXPERIMENTS = {
    'LGB_LogRatio':  {'algo':'lgb',      'feats':E6_BASE_SM,  'size':[0,1], 'multi_h':False},
    'XGB_LogRatio':  {'algo':'xgb',      'feats':E6_BASE_SM,  'size':[0,1], 'multi_h':False},
    'CB_LogRatio':   {'algo':'catboost', 'feats':E6_BASE_SM,  'size':[0,1], 'multi_h':False},
    'LGB_MultiH':    {'algo':'lgb',      'feats':E6_MULTI_SM, 'size':[0,1], 'multi_h':True},
    'LGB_LargeCol':  {'algo':'lgb',      'feats':E6_BASE_LC,  'size':[2,3], 'multi_h':False},
    'LGB_MultiH_LC': {'algo':'lgb',      'feats':E6_MULTI_LC, 'size':[2,3], 'multi_h':True},
    'RF_LogRatio':   {'algo':'rf',  'feats':E6_BASE_SM,  'size':[0,1], 'multi_h':False},
    'RF_MultiH':     {'algo':'rf',  'feats':E6_MULTI_SM, 'size':[0,1], 'multi_h':True},
    'XGB_LargeCol':  {'algo':'xgb', 'feats':E6_BASE_LC,  'size':[2,3], 'multi_h':False},
    'RF_LargeCol':   {'algo':'rf',  'feats':E6_BASE_LC,  'size':[2,3], 'multi_h':False},
    'RF_MultiH_LC':  {'algo':'rf',  'feats':E6_MULTI_LC, 'size':[2,3], 'multi_h':True},
    # OLS-augmented: gradient/RF models receive OLS linear prediction as additional feature
    'LGB_MultiH_OLS':    {'algo':'lgb', 'feats':E6_MULTI_SM_OLS, 'size':[0,1], 'multi_h':True},
    'RF_MultiH_OLS':     {'algo':'rf',  'feats':E6_MULTI_SM_OLS, 'size':[0,1], 'multi_h':True},
    'LGB_MultiH_LC_OLS': {'algo':'lgb', 'feats':E6_MULTI_LC_OLS, 'size':[2,3], 'multi_h':True},
    'RF_MultiH_LC_OLS':  {'algo':'rf',  'feats':E6_MULTI_LC_OLS, 'size':[2,3], 'multi_h':True},
}

print(f'Features SM ({len(E6_BASE_SM)}): {E6_BASE_SM}')
print(f'Features LC ({len(E6_BASE_LC)}): {E6_BASE_LC}')
print(f'Experimentos: {list(EXPERIMENTS.keys())}')

Dataset: 1,158 rows | 34 minas
Features SM (9): ['Company_Size', 'Mine_Size', 'Prod_Lag1', 'Tendencia_5y', 'Prod_pct_change', 'Mine_age', 'Mine_share', 'Is_Pandemic_Target', 'Cu_regime']
Features LC (10): ['Company_Size', 'Mine_Size', 'Prod_Lag1', 'Tendencia_5y', 'Prod_pct_change', 'Mine_age', 'Mine_share', 'Is_Pandemic_Target', 'Cu_regime', 'Capital_Stock_Lag1']
Experimentos: ['LGB_LogRatio', 'XGB_LogRatio', 'CB_LogRatio', 'LGB_MultiH', 'LGB_LargeCol', 'LGB_MultiH_LC', 'RF_LogRatio', 'RF_MultiH', 'XGB_LargeCol', 'RF_LargeCol', 'RF_MultiH_LC', 'LGB_MultiH_OLS', 'RF_MultiH_OLS', 'LGB_MultiH_LC_OLS', 'RF_MultiH_LC_OLS']


## 4. Pre-cómputo SARIMAX

In [4]:
ts_records = {}; sarimax_by_origin = {}
print('Pre-computando SARIMAX por origen...')
for origin_year in ROLLING_ORIGINS:
    df_tr = df_raw[df_raw['Anio']<=origin_year].copy()
    sarimax = SARIMAXPerMine(); sarimax.fit(df_tr)
    sarimax_by_origin[origin_year] = sarimax
    prod_origin = df_raw[df_raw['Anio']==origin_year].groupby('Match_Key')['Produccion'].mean()
    prod_actual = df_raw.set_index(['Match_Key','Anio'])['Produccion']
    for h in HORIZONS:
        fy = origin_year+h
        if fy>BASE_YEAR: continue
        for mine in MINES:
            try: actual = float(prod_actual.loc[(mine,fy)])
            except: continue
            naive = float(prod_origin.get(mine, np.nan))
            if pd.isna(naive) or naive==0: continue
            pred_s = sarimax.predict_h(mine,h)
            if pd.isna(pred_s): continue
            ne=abs(actual-naive); me=abs(actual-pred_s)
            key = (origin_year,mine,h)
            ts_records[key] = {'actual':actual,'naive':naive,'sarimax_pred':pred_s,
                               'ne':ne,'me_sarimax':me,'beats_sarimax':int(me<ne)}
    print(f'  {origin_year}', end=' ', flush=True)
print()

sarimax_wr = np.mean([v['beats_sarimax'] for v in ts_records.values()])
print(f'SARIMAX WR global = {sarimax_wr*100:.1f}%')
for h in H_FOCUS:
    h_wr = np.mean([v['beats_sarimax'] for k,v in ts_records.items() if k[2]==h])
    print(f'  H+{h}: {h_wr*100:.1f}%')

Pre-computando SARIMAX por origen...


  2010 

  2011 

  2012

  2013 

...

  2015 

  2016 

  2017

  2018 


SARIMAX WR global = 48.5%
  H+5: 49.1%
  H+6: 53.3%
  H+7: 51.5%


## 4b. Rolling OLS (expanding window) por Anio

Para cada anio `t` (desde el primer anio con suficientes datos hasta el ultimo origin),
se ajusta un OLS usando **solo datos disponibles hasta ese anio**: `Anio <= t AND Target_Year <= t`.
La prediccion OLS para `(mine, t, h)` es genuinamente out-of-sample: el Target `t+h` no fue visto
al momento de ajustar el OLS en `t`.

Las predicciones se almacenan en `ols_pred_cache[(anio, mine, h)]`.
El helper `_inject_ols_pred_rolling(df)` inyecta OLS_Pred via lookup en el cache.


In [5]:
# Rolling OLS (expanding window) — verdaderamente sin fuga de datos
# Para cada anio t: OLS ajustado solo en datos Anio<=t, Target_Year<=t
# La prediccion para (mine, t, h) predice Target_Year=t+h > t → out-of-sample
from sklearn.linear_model import LinearRegression as _OLS

print('Computando rolling OLS por anio (expanding window)...')
ols_pred_cache = {}  # (anio, mine, h) -> OLS prediction en LogRatio space

# Rango: primer anio con suficiente historia hasta el ultimo rolling origin
# Skip anios muy tempranos donde el OLS no tendra suficientes datos
_MIN_OLS_YEAR = 1995
_MAX_OLS_YEAR = max(ROLLING_ORIGINS)  # 2018

for _t in range(_MIN_OLS_YEAR, _MAX_OLS_YEAR + 1):
    # Mine_Size en el anio t (rolling 6y antes de t) — correcto para OLS en t
    _ms_t = compute_mine_size(df_raw, _t)
    _dft = df_feats.copy()
    _dft['Mine_Size'] = _dft['Match_Key'].map(_ms_t).fillna(1).astype(int)

    # Construir datos de entrenamiento OLS: solo datos completamente observados en t
    _fsm, _flc = [], []
    for _h in HORIZONS:
        _d = _dft.copy()
        _d['Target']             = _d.groupby('Match_Key')['Produccion'].shift(-_h)
        _d['Target_Year']        = _d['Anio'] + _h
        _d['Horizonte_feat']     = _h
        _d['Is_Pandemic_Target'] = _d['Target_Year'].isin(PANDEMIC_YEARS).astype(int)
        # Strict: Target_Year <= t (el target fue observado antes de la prediccion en t)
        _sub = _d[(_d['Anio'] <= _t) & (_d['Target_Year'] <= _t) & (_d['Prod_Lag1'] > 0)]
        _fsm.append(_sub[_sub['Mine_Size'].isin([0,1])].dropna(
            subset=E6_MULTI_SM + ['Target', 'Produccion']))
        _flc.append(_sub[_sub['Mine_Size'].isin([2,3])].dropna(
            subset=E6_MULTI_LC + ['Target', 'Produccion']))

    _trsm = pd.concat(_fsm, ignore_index=True)
    _trlc = pd.concat(_flc, ignore_index=True)
    _ols_sm_t = _ols_lc_t = None
    if len(_trsm) >= 10:
        _ysm = np.clip(np.log((_trsm['Target']+1e-6)/(_trsm['Produccion']+1e-6)), -3, 3).values
        _ols_sm_t = _OLS().fit(_trsm[E6_MULTI_SM].fillna(0).values, _ysm)
    if len(_trlc) >= 10:
        _ylc = np.clip(np.log((_trlc['Target']+1e-6)/(_trlc['Produccion']+1e-6)), -3, 3).values
        _ols_lc_t = _OLS().fit(_trlc[E6_MULTI_LC].fillna(0).values, _ylc)

    # Generar predicciones OLS para todos (mine, h) en el anio t
    # Estas predicciones son forward-looking desde t: Target=t+h NO fue visto al ajustar OLS
    for _h in HORIZONS:
        _dh = _dft.copy()
        _dh['Target']             = _dh.groupby('Match_Key')['Produccion'].shift(-_h)
        _dh['Target_Year']        = _dh['Anio'] + _h
        _dh['Horizonte_feat']     = _h
        _dh['Is_Pandemic_Target'] = _dh['Target_Year'].isin(PANDEMIC_YEARS).astype(int)
        _rows_t = _dh[(_dh['Anio'] == _t) & (_dh['Prod_Lag1'] > 0)]
        for _, _row in _rows_t.iterrows():
            _mine = _row['Match_Key']
            _ms   = int(_row['Mine_Size'])
            _ols_mod = _ols_sm_t if _ms <= 1 else _ols_lc_t
            if _ols_mod is None: continue
            _feats = E6_MULTI_SM if _ms <= 1 else E6_MULTI_LC
            try:
                _x = _row[_feats].fillna(0).values.reshape(1, -1)
                ols_pred_cache[(_t, _mine, _h)] = float(
                    np.clip(_ols_mod.predict(_x), -3, 3)[0])
            except:
                pass

    if _t % 5 == 0 or _t == _MAX_OLS_YEAR:
        _n = sum(1 for k in ols_pred_cache if k[0] == _t)
        print(f'  {_t}: SM n={len(_trsm):4d}, LC n={len(_trlc):3d} | preds this year={_n}')

print(f'Rolling OLS cache: {len(ols_pred_cache):,} entradas')

# Vectorized injection helper — usa el cache rolling (sin fuga en training rows)
def _inject_ols_pred_rolling(df):
    """Add OLS_Pred via rolling cache lookup. Rows with no cache entry get 0.0."""
    df = df.copy()
    _keys = list(zip(df['Anio'].astype(int),
                     df['Match_Key'],
                     df['Horizonte_feat'].astype(int)))
    df['OLS_Pred'] = [ols_pred_cache.get(_k, 0.0) for _k in _keys]
    return df

print('Helper _inject_ols_pred_rolling listo.')

Computando rolling OLS por anio (expanding window)...
  1995: SM n=   0, LC n=220 | preds this year=70


  2000: SM n=   0, LC n=528 | preds this year=112


  2005: SM n=   0, LC n=1022 | preds this year=119


  2010: SM n=   2, LC n=1610 | preds this year=119


  2015: SM n= 317, LC n=1938 | preds this year=231
  2018: SM n= 606, LC n=2089 | preds this year=217
Rolling OLS cache: 3,185 entradas
Helper _inject_ols_pred_rolling listo.


## 5. Optuna Tuning y Loop ML Rolling-Origin

In [6]:
# CatBoost removed from ensemble — skip training to save compute
EXPERIMENTS.pop('CB_LogRatio', None)
print(f"Experiments to run: {list(EXPERIMENTS.keys())}")


Experiments to run: ['LGB_LogRatio', 'XGB_LogRatio', 'LGB_MultiH', 'LGB_LargeCol', 'LGB_MultiH_LC', 'RF_LogRatio', 'RF_MultiH', 'XGB_LargeCol', 'RF_LargeCol', 'RF_MultiH_LC', 'LGB_MultiH_OLS', 'RF_MultiH_OLS', 'LGB_MultiH_LC_OLS', 'RF_MultiH_LC_OLS']


In [7]:
ml_records   = []
optuna_params = {}
TUNE_ORIGIN = 2010   # Fixed at first rolling origin — no hyperparameter leakage

print(f'Loop ML v7 — {len(EXPERIMENTS)} experimentos × {len(ROLLING_ORIGINS)} orígenes × {len(HORIZONS)} horizontes\n')

for exp_name, cfg in EXPERIMENTS.items():
    feat_list = cfg['feats']
    algo      = cfg['algo']
    size_filt = cfg['size']
    multi_h   = cfg['multi_h']
    n_trials  = LGB_TRIALS if algo=='lgb' else (XGB_TRIALS if algo=='xgb' else (RF_TRIALS if algo=='rf' else CB_TRIALS))
    print(f'\n=== {exp_name} | algo={algo} | multi_h={multi_h} | {n_trials} trials ===')

    ms_tune = compute_mine_size(df_raw, TUNE_ORIGIN)
    df_feats['Mine_Size'] = df_feats['Match_Key'].map(ms_tune).fillna(1).astype(int)

    if multi_h:
        tune_frames = []
        for h_t in HORIZONS:
            df_ht = df_feats.copy()
            df_ht['Target']             = df_ht.groupby('Match_Key')['Produccion'].shift(-h_t)
            df_ht['Target_Year']        = df_ht['Anio'] + h_t
            df_ht['Horizonte_feat']     = h_t
            df_ht['Is_Pandemic_Target'] = df_ht['Target_Year'].isin(PANDEMIC_YEARS).astype(int)
            sub = df_ht[(df_ht['Anio']<=TUNE_ORIGIN)&
                        (df_ht['Mine_Size'].isin(size_filt))&(df_ht['Prod_Lag1']>0)
                       ].dropna(subset=[f for f in feat_list if f!='OLS_Pred']+['Target','Produccion'])
            tune_frames.append(sub)
        tune_df = pd.concat(tune_frames, ignore_index=True)
        if 'OLS_Pred' in feat_list:
            tune_df = _inject_ols_pred_rolling(tune_df)
            tune_df = tune_df.dropna(subset=['OLS_Pred'])
    else:
        df_h_tune = df_feats.copy()
        df_h_tune['Target']             = df_h_tune.groupby('Match_Key')['Produccion'].shift(-TUNE_H)
        df_h_tune['Target_Year']        = df_h_tune['Anio'] + TUNE_H
        df_h_tune['Is_Pandemic_Target'] = df_h_tune['Target_Year'].isin(PANDEMIC_YEARS).astype(int)
        tune_df = df_h_tune[(df_h_tune['Anio']<=TUNE_ORIGIN)&
                            (df_h_tune['Mine_Size'].isin(size_filt))&
                            (df_h_tune['Prod_Lag1']>0)].dropna(subset=[f for f in feat_list if f!='OLS_Pred']+['Target','Produccion'])

    if len(tune_df) >= 8:
        y_t = np.clip(np.log((tune_df['Target']+1e-6)/(tune_df['Produccion']+1e-6)).values,-3,3)
        tuner = make_optuna_tuner(algo, n_trials)
        params = tuner(tune_df[feat_list].fillna(0).values, y_t)
        optuna_params[exp_name] = params
        print(f'  Optuna OK (origin={TUNE_ORIGIN}, n={len(tune_df)}): {params}')
    else:
        print(f'  Optuna SKIP: tune_df too small (n={len(tune_df)}, origin={TUNE_ORIGIN})')
        optuna_params[exp_name] = {}

    for origin_year in ROLLING_ORIGINS:
        ms_map = compute_mine_size(df_raw, origin_year)
        df_feats['Mine_Size'] = df_feats['Match_Key'].map(ms_map).fillna(1).astype(int)
        prod_origin = df_raw[df_raw['Anio']==origin_year].groupby('Match_Key')['Produccion'].mean()

        if multi_h:
            train_frames = []
            for h_tr in HORIZONS:
                df_ht = df_feats.copy()
                df_ht['Target']             = df_ht.groupby('Match_Key')['Produccion'].shift(-h_tr)
                df_ht['Target_Year']        = df_ht['Anio'] + h_tr
                df_ht['Horizonte_feat']     = h_tr
                df_ht['Is_Pandemic_Target'] = df_ht['Target_Year'].isin(PANDEMIC_YEARS).astype(int)
                sub = df_ht[(df_ht['Anio']<=origin_year)&(df_ht['Target_Year']<=origin_year)&
                            (df_ht['Mine_Size'].isin(size_filt))&(df_ht['Prod_Lag1']>0)
                           ].dropna(subset=[f for f in feat_list if f!='OLS_Pred']+['Target','Produccion'])
                train_frames.append(sub)
            train = pd.concat(train_frames, ignore_index=True)
            # ── OLS_Pred injection (train) ──────────────────────────────────────
            if 'OLS_Pred' in feat_list:
                train = _inject_ols_pred_rolling(train)
        else:
            train = None

        for h in HORIZONS:
            fy = origin_year+h
            if fy>BASE_YEAR: continue

            if not multi_h:
                df_h = df_feats.copy()
                df_h['Target']             = df_h.groupby('Match_Key')['Produccion'].shift(-h)
                df_h['Target_Year']        = df_h['Anio']+h
                df_h['Is_Pandemic_Target'] = df_h['Target_Year'].isin(PANDEMIC_YEARS).astype(int)
                train = df_h[(df_h['Anio']<=origin_year)&(df_h['Target_Year']<=origin_year)&
                             (df_h['Mine_Size'].isin(size_filt))&(df_h['Prod_Lag1']>0)
                            ].dropna(subset=[f for f in feat_list if f!='OLS_Pred']+['Target','Produccion'])
                df_test = df_h
            else:
                df_h = df_feats.copy()
                df_h['Target']             = df_h.groupby('Match_Key')['Produccion'].shift(-h)
                df_h['Target_Year']        = df_h['Anio']+h
                df_h['Is_Pandemic_Target'] = df_h['Target_Year'].isin(PANDEMIC_YEARS).astype(int)
                df_h['Horizonte_feat']     = h
                df_test = df_h

            test = df_test[(df_test['Anio']==origin_year)&(df_test['Target_Year']==fy)&
                           (df_test['Mine_Size'].isin(size_filt))
                          ].dropna(subset=[f for f in feat_list if f!='OLS_Pred']+['Target','Produccion'])
            # ── OLS_Pred injection (test) ───────────────────────────────────────
            if 'OLS_Pred' in feat_list and len(test) > 0:
                test = _inject_ols_pred_rolling(test)
                test = test.dropna(subset=['OLS_Pred'])

            if len(train)<10 or len(test)==0: continue

            y_tr = np.clip(np.log((train['Target']+1e-6)/(train['Produccion']+1e-6)).values,-3,3)
            X_tr = train[feat_list].fillna(0).values

            if algo == 'lgb':   model = LGBWrapper(**optuna_params[exp_name])
            elif algo == 'xgb': model = XGBWrapper(**optuna_params[exp_name])
            elif algo == 'rf':  model = RFWrapper(**optuna_params[exp_name])
            else:               model = CBWrapper(**optuna_params[exp_name])

            try:
                if algo == 'catboost':
                    ci = [i for i,f in enumerate(feat_list) if f in ('Company_Size','Mine_Size')]
                    model.fit(X_tr, y_tr, cat_features=ci)
                else:
                    model.fit(X_tr, y_tr)
            except: model = None
            if model is None: continue

            for _, row in test.iterrows():
                mine   = row['Match_Key']
                actual = row['Target']
                naive  = float(prod_origin.get(mine, np.nan))
                if pd.isna(naive) or naive==0: continue
                ne = abs(actual-naive)
                x  = np.array(row[feat_list].fillna(0)).reshape(1,-1)
                origin_prod = row['Produccion']
                try: raw_pred = float(model.predict(x)[0])
                except: continue
                pred = max(0, np.exp(raw_pred) * (origin_prod + 1e-6))
                me   = abs(actual-pred)
                mape = abs(actual-pred)/(abs(actual)+1)*100 if actual>0 else np.nan
                sarimax_pred = sarimax_by_origin[origin_year].predict_h(mine, h)
                if pd.isna(sarimax_pred): sarimax_pred = naive
                ml_records.append({
                    'Exp':exp_name,'Origin':origin_year,'Horizonte':h,
                    'ForecastYear':fy,'Mine':mine,'Actual':actual,
                    'Pred':pred,'Naive_Pred':naive,'SARIMAX_Pred':sarimax_pred,
                    'Company_Size':int(row['Company_Size']),'Mine_Size':int(row['Mine_Size']),
                    'Origin_Prod':origin_prod,'Model_Error':me,'Naive_Error':ne,
                    'Beats_Naive':int(me<ne),'MAPE':mape,
                    'Is_Pandemic_Target':int(fy in PANDEMIC_YEARS),
                })
        print(f'  {origin_year}', end=' ', flush=True)
    print()

df_ml = pd.DataFrame(ml_records)
print(f'\nTotal registros ML: {len(df_ml):,}')

Loop ML v7 — 14 experimentos × 9 orígenes × 7 horizontes


=== LGB_LogRatio | algo=lgb | multi_h=False | 50 trials ===


  Optuna SKIP: tune_df too small (n=4, origin=2010)
  2010 

  2011 

  2012 

  2013 

...

  2015 

  2016 

  2017 

  2018 



Total registros ML: 9,555


## 6. Ensemble Post-hoc (LGB + SARIMAX)

In [8]:
ensemble_records = []

# SmallMed ensemble: auto-select best base model by WR on H_FOCUS
sm_candidates = ['LGB_LogRatio', 'XGB_LogRatio', 'LGB_MultiH', 'RF_LogRatio', 'RF_MultiH']
best_sm_exp = max(
    [e for e in sm_candidates if e in df_ml['Exp'].values],
    key=lambda e: -df_ml[(df_ml['Exp']==e) & df_ml['Horizonte'].isin(H_FOCUS)]['MAPE'].median()
)
print(f'  Best SmallMed base: {best_sm_exp} (WR_focus={100*df_ml[(df_ml["Exp"]==best_sm_exp)&df_ml["Horizonte"].isin(H_FOCUS)]["Beats_Naive"].mean():.1f}%)')
base_sm = df_ml[df_ml['Exp'] == best_sm_exp].copy()
for alpha_name, alpha_val in [('Ens_5050', 0.5), ('Ens_7030', 0.7), ('Ens_3070', 0.3)]:
    for _, row in base_sm.iterrows():
        sp = row['SARIMAX_Pred']
        pred_ens = alpha_val * row['Pred'] + (1-alpha_val) * sp
        me_ens = abs(row['Actual'] - pred_ens)
        ensemble_records.append({
            'Exp':alpha_name,'Origin':row['Origin'],'Horizonte':row['Horizonte'],
            'ForecastYear':row['ForecastYear'],'Mine':row['Mine'],'Actual':row['Actual'],
            'Pred':pred_ens,'Naive_Pred':row['Naive_Pred'],'SARIMAX_Pred':sp,
            'Company_Size':row['Company_Size'],'Mine_Size':row['Mine_Size'],
            'Model_Error':me_ens,'Naive_Error':row['Naive_Error'],
            'Beats_Naive':int(me_ens<row['Naive_Error']),
            'MAPE':abs(row['Actual']-pred_ens)/(abs(row['Actual'])+1)*100 if row['Actual']>0 else np.nan,
        })

# LargeCol ensemble: auto-select best base model by WR on H_FOCUS
lc_candidates = ['LGB_LargeCol', 'XGB_LargeCol', 'RF_LargeCol', 'LGB_MultiH_LC', 'RF_MultiH_LC']
best_lc_exp = max(
    [e for e in lc_candidates if e in df_ml['Exp'].values],
    key=lambda e: -df_ml[(df_ml['Exp']==e) & df_ml['Horizonte'].isin(H_FOCUS)]['MAPE'].median()
)
print(f'  Best LargeCol base: {best_lc_exp} (WR_focus={100*df_ml[(df_ml["Exp"]==best_lc_exp)&df_ml["Horizonte"].isin(H_FOCUS)]["Beats_Naive"].mean():.1f}%)')
lc_base = df_ml[df_ml['Exp'] == best_lc_exp].copy()
for alpha_name, alpha_val in [('Ens_LC_5050', 0.5), ('Ens_LC_7030', 0.7), ('Ens_LC_3070', 0.3)]:
    for _, row in lc_base.iterrows():
        sp = row['SARIMAX_Pred']
        pred_ens = alpha_val * row['Pred'] + (1-alpha_val) * sp
        me_ens = abs(row['Actual'] - pred_ens)
        ensemble_records.append({
            'Exp':alpha_name,'Origin':row['Origin'],'Horizonte':row['Horizonte'],
            'ForecastYear':row['ForecastYear'],'Mine':row['Mine'],'Actual':row['Actual'],
            'Pred':pred_ens,'Naive_Pred':row['Naive_Pred'],'SARIMAX_Pred':sp,
            'Company_Size':row['Company_Size'],'Mine_Size':row['Mine_Size'],
            'Model_Error':me_ens,'Naive_Error':row['Naive_Error'],
            'Beats_Naive':int(me_ens<row['Naive_Error']),
            'MAPE':abs(row['Actual']-pred_ens)/(abs(row['Actual'])+1)*100 if row['Actual']>0 else np.nan,
        })

# SARIMAX standalone (all mines)
_ms_cache = {oy: compute_mine_size(df_raw, oy) for oy in ROLLING_ORIGINS}
for key, v in ts_records.items():
    origin_year, mine, h = key
    ms_map = _ms_cache[origin_year]
    ensemble_records.append({
        'Exp':'SARIMAX','Origin':origin_year,'Horizonte':h,
        'ForecastYear':origin_year+h,'Mine':mine,'Actual':v['actual'],
        'Pred':v['sarimax_pred'],'Naive_Pred':v['naive'],
        'Company_Size':COMPANY_SIZE_MAP.get(mine,1),'Mine_Size':ms_map.get(mine,1),
        'Model_Error':v['me_sarimax'],'Naive_Error':v['ne'],'Beats_Naive':v['beats_sarimax'],
        'MAPE':abs(v['actual']-v['sarimax_pred'])/(abs(v['actual'])+1)*100 if v['actual']>0 else np.nan,
    })

# SM Super-ensemble: 50% LGB_MultiH + 50% RF_MultiH → 30% ML + 70% SARIMAX
if 'LGB_MultiH' in df_ml['Exp'].values and 'RF_MultiH' in df_ml['Exp'].values:
    _lgb_s = df_ml[df_ml['Exp']=='LGB_MultiH'][
        ['Origin','Horizonte','Mine','ForecastYear','Actual','Pred',
         'Naive_Pred','SARIMAX_Pred','Company_Size','Mine_Size','Naive_Error']].copy()
    _rf_s  = df_ml[df_ml['Exp']=='RF_MultiH'][
        ['Origin','Horizonte','Mine','Pred']].rename(columns={'Pred':'Pred_RF'})
    _super = _lgb_s.merge(_rf_s, on=['Origin','Horizonte','Mine'], how='inner')
    for _, row in _super.iterrows():
        ml_pred  = 0.5 * row['Pred'] + 0.5 * row['Pred_RF']
        sp       = row['SARIMAX_Pred']
        pred_ens = 0.3 * ml_pred + 0.7 * sp
        me_ens   = abs(row['Actual'] - pred_ens)
        ensemble_records.append({
            'Exp':'Ens_Super_Annual',
            'Origin':row['Origin'],'Horizonte':row['Horizonte'],
            'ForecastYear':row['ForecastYear'],'Mine':row['Mine'],
            'Actual':row['Actual'],'Pred':pred_ens,
            'Naive_Pred':row['Naive_Pred'],'SARIMAX_Pred':sp,
            'Company_Size':row['Company_Size'],'Mine_Size':row['Mine_Size'],
            'Model_Error':me_ens,'Naive_Error':row['Naive_Error'],
            'Beats_Naive':int(me_ens<row['Naive_Error']),
            'MAPE':abs(row['Actual']-pred_ens)/(abs(row['Actual'])+1)*100 if row['Actual']>0 else np.nan,
        })
    print(f'  Ens_Super_Annual: {len(_super)} rows (0.5*LGB_MultiH + 0.5*RF_MultiH → 30/70 SARIMAX)')
else:
    print('  WARNING: LGB_MultiH or RF_MultiH missing — Ens_Super_Annual skipped')

df_ens = pd.DataFrame(ensemble_records)
df_all = pd.concat([df_ml, df_ens], ignore_index=True)

# Ens_Segmentado: SM → Ens_3070, Large [2] → Ens_LC_3070, Colossal [3] + problem mines → SARIMAX
SARIMAX_ONLY_MINES = {'caserones'}
df_sm_ens  = df_all[df_all['Exp']=='Ens_Super_Annual'].copy()  # SM: 0.5*LGB + 0.5*RF + SARIMAX
df_lc_ens  = df_all[df_all['Exp']=='Ens_LC_3070'].copy()
df_sar_all = df_all[df_all['Exp']=='SARIMAX'].copy().set_index(['Mine','Origin','Horizonte'])

def _sarimax_override(d):
    key = (d['Mine'], d['Origin'], d['Horizonte'])
    if key in df_sar_all.index:
        row_s = df_sar_all.loc[key]
        pred_s = float(row_s['Pred'].iloc[0] if hasattr(row_s['Pred'],'iloc') else row_s['Pred'])
        d['Pred'] = pred_s
        d['Model_Error'] = abs(d['Actual'] - pred_s)
        d['Beats_Naive'] = int(d['Model_Error'] < d['Naive_Error'])
    return d

seg_records = []
for _, row in df_sm_ens.iterrows():
    d = row.to_dict(); d['Exp'] = 'Ens_Segmentado'
    if d['Mine'] in SARIMAX_ONLY_MINES:
        d = _sarimax_override(d)
    seg_records.append(d)
for _, row in df_lc_ens.iterrows():
    d = row.to_dict(); d['Exp'] = 'Ens_Segmentado'
    if d['Mine_Size'] == 3 or d['Mine'] in SARIMAX_ONLY_MINES:
        d = _sarimax_override(d)
    seg_records.append(d)
df_seg = pd.DataFrame(seg_records)
df_all = pd.concat([df_all, df_seg], ignore_index=True)

# ── Ens_Adaptive: per-mine optimal ML weight (minimizes MAE over validation) ──
from scipy.optimize import minimize_scalar

adaptive_records_ann = []
for _mine in df_all['Mine'].unique():
    _is_lc  = df_all[(df_all['Mine']==_mine) & df_all['Mine_Size'].isin([2,3])].shape[0] > 0
    _base_e = 'Ens_LC_3070' if _is_lc else 'Ens_3070'
    _sub    = df_all[(df_all['Exp']==_base_e) & (df_all['Mine']==_mine) & (df_all['Actual']>0)].copy()
    if len(_sub) < 5:
        _alpha = 0.3
    else:
        def _mae_fn(a, s=_sub):
            return float(np.mean(np.abs(s['Actual'] - (a * s['Pred'] + (1-a) * s['Naive_Pred']))))
        _res   = minimize_scalar(_mae_fn, bounds=(0.0, 1.0), method='bounded')
        _alpha = round(float(_res.x), 3)
    for _, _row in _sub.iterrows():
        _p  = max(0.0, _alpha * _row['Pred'] + (1 - _alpha) * _row['Naive_Pred'])
        _me = abs(_row['Actual'] - _p)
        adaptive_records_ann.append({
            **{k: _row[k] for k in ['Origin','Horizonte','ForecastYear','Mine','Actual',
                                    'Naive_Pred','Company_Size','Mine_Size','Naive_Error',
                                    'SARIMAX_Pred']},
            'Exp': 'Ens_Adaptive',
            'Pred': _p,
            'Model_Error': _me,
            'Beats_Naive': int(_me < _row['Naive_Error']),
            'MAPE': abs(_row['Actual']-_p)/(abs(_row['Actual'])+1)*100 if _row['Actual']>0 else np.nan,
            'Alpha_Mine': _alpha,
        })

df_adaptive_ann = pd.DataFrame(adaptive_records_ann)
df_all = pd.concat([df_all, df_adaptive_ann], ignore_index=True)

print("Mine-specific adaptive alphas (annual):")
print(df_adaptive_ann.groupby('Mine')['Alpha_Mine'].first().sort_values().to_string())

df_all.to_csv(f'{EXPORT_DIR}/predicciones_anuales_baseline_extendido.csv', index=False)
print(f'Total registros: {len(df_all):,}  → guardado en {EXPORT_DIR}/predicciones_anuales_baseline_extendido.csv')

  Best SmallMed base: RF_MultiH (WR_focus=63.5%)
  Best LargeCol base: RF_MultiH_LC (WR_focus=49.5%)
  Ens_Super_Annual: 294 rows (0.5*LGB_MultiH + 0.5*RF_MultiH → 30/70 SARIMAX)


Mine-specific adaptive alphas (annual):
Mine
los pelambres                    0.000
andina                           0.000
lomas bayas                      0.000
radomiro tomic                   0.000
caserones                        0.000
centinela_centinela_óxidos_      0.000
chuquicamata                     0.000
collahuasi                       0.000
escondida                        0.000
michilla                         0.000
los bronces                      0.110
gabriela mistral                 0.482
antucoya                         0.604
zaldivar                         0.671
cerro colorado                   0.769
sierra gorda                     0.930
ministro hales                   1.000
salvador                         1.000
andacollo                        1.000
centinela_centinela_sulfuros_    1.000
capstone copper (4)              1.000
candelaria                       1.000
el teniente                      1.000
el abra                          1.000
Total registros: 17

## 7. Resultados

In [9]:
SM_EXPS = ['LGB_LogRatio','XGB_LogRatio','LGB_MultiH','RF_MultiH','Ens_5050','Ens_7030','Ens_3070','Ens_Super_Annual']
LC_EXPS = ['LGB_LargeCol','LGB_MultiH_LC','Ens_LC_5050','Ens_LC_7030','Ens_LC_3070']
ALL_EXPS = SM_EXPS + LC_EXPS + ['SARIMAX','Ens_Segmentado','Ens_Adaptive']

print('='*90)
print('  RESULTADOS v7 — SmallMed + LargeColossal + Ens_Segmentado + Ens_Adaptive')
print('='*90)
print(f'\n  {"Estrategia":<22} {"Seg":>4} {"Skill%":>8} {"WR%":>6} {"M50":>4} '
      f'{"H4":>6} {"H5":>6} {"H6":>6} {"H7":>6} {"H57":>7}')
print(f'  {"-"*82}')

resumen_rows = []
for exp_name in ALL_EXPS:
    sub = df_all[df_all['Exp']==exp_name].dropna(subset=['Beats_Naive'])
    if len(sub)==0: continue
    wr    = 100*sub['Beats_Naive'].mean()
    skill = (sub['Naive_Error'].mean()-sub['Model_Error'].mean())/sub['Naive_Error'].mean()*100
    n50   = int((sub.groupby('Mine')['Beats_Naive'].mean()>=0.5).sum())
    wr_h  = {h:round(100*sub[sub['Horizonte']==h]['Beats_Naive'].mean(),1) for h in HORIZONS}
    h57   = np.mean([wr_h.get(h,np.nan) for h in [5,6,7]])
    seg   = 'LC' if exp_name in LC_EXPS else ('ALL' if exp_name in ['SARIMAX','Ens_Segmentado','Ens_Adaptive'] else 'SM')
    em    = '=>' if h57>=55 else ('~>' if h57>=50 else '  ')
    print(f'  {em}{exp_name:<20} {seg:>4} {skill:>+8.1f} {wr:>6.1f} {n50:>4} '
          f'{wr_h.get(4,0):>6.1f} {wr_h.get(5,0):>6.1f} {wr_h.get(6,0):>6.1f} '
          f'{wr_h.get(7,0):>6.1f} {h57:>7.1f}')
    resumen_rows.append({'Exp':exp_name,'Segment':seg,'Skill_%':round(skill,1),'WR_%':round(wr,1),
        'Minas_50':n50,'H57_avg':round(h57,1),**{f'H{h}':wr_h.get(h,np.nan) for h in HORIZONS}})

pd.DataFrame(resumen_rows).to_csv(f'{EXPORT_DIR}/resumen_annual_v7.csv', index=False)
best_sm  = max([r for r in resumen_rows if r['Segment']=='SM'],  key=lambda r: r['H57_avg'], default=None)
best_lc  = max([r for r in resumen_rows if r['Segment']=='LC'],  key=lambda r: r['H57_avg'], default=None)
best_seg = next((r for r in resumen_rows if r['Exp']=='Ens_Segmentado'), None)
if best_sm:  print(f'\n  Best SM:  {best_sm["Exp"]} H57={best_sm["H57_avg"]:.1f}%')
if best_lc:  print(f'  Best LC:  {best_lc["Exp"]} H57={best_lc["H57_avg"]:.1f}%')
if best_seg: print(f'  Ens_Seg:  H57={best_seg["H57_avg"]:.1f}%')

# ── Scoreboard per-mine ────────────────────────────────────────────────────────
df_b     = df_all[(df_all['Exp']=='Ens_Segmentado') &
                  (df_all['Horizonte'].isin(H_FOCUS))].dropna(subset=['Beats_Naive'])
mine_agg = df_b.groupby('Mine').agg(
    WR=('Beats_Naive','mean'), MAE_M=('Model_Error','mean'),
    MAE_N=('Naive_Error','mean'), MAPE=('MAPE','mean'), MS=('Mine_Size','first')
).reset_index()
mine_agg['MASE']  = (mine_agg['MAE_M']/mine_agg['MAE_N']).round(3)
mine_agg['Skill'] = ((mine_agg['MAE_N']-mine_agg['MAE_M'])/mine_agg['MAE_N']*100).round(1)

# MdAPE Skill
_mn = df_b.copy()
_mn['Naive_MAPE'] = ((_mn['Actual']-_mn['Naive_Pred']).abs()/(_mn['Actual'].abs()+1)*100)
mine_agg = mine_agg.join(_mn.groupby('Mine')['Naive_MAPE'].median().rename('MdAPE_N'), on='Mine')
mine_agg = mine_agg.join(df_b.groupby('Mine')['MAPE'].median().rename('MdAPE_M'),       on='Mine')
mine_agg['MdAPE_Skill'] = ((mine_agg['MdAPE_N']-mine_agg['MdAPE_M'])/mine_agg['MdAPE_N']*100).round(1)

# Winsorized Skill (recorta top/bottom 5% de errores por mina)
def _wskill(grp, trim=0.05):
    em = np.sort(grp['Model_Error'].values)
    en = np.sort(grp['Naive_Error'].values)
    k  = max(1, int(len(em)*trim))
    em_w = em[k:-k] if len(em) > 2*k else em
    en_w = en[k:-k] if len(en) > 2*k else en
    return round((en_w.mean()-em_w.mean())/en_w.mean()*100, 1) if en_w.mean() != 0 else np.nan
mine_agg = mine_agg.join(df_b.groupby('Mine').apply(_wskill).rename('WSkill'), on='Mine')

mine_agg['Size_Label'] = mine_agg['MS'].map(SIZE_LBL)
mine_agg = mine_agg.sort_values('MdAPE_Skill', ascending=False)
mine_agg.to_csv(f'{EXPORT_DIR}/scoreboard_annual_v7.csv', index=False)

print(f'\n  {"Mine":<42} {"Size":>8} {"Skill%":>8} {"WSkill%":>9} {"MdAPE_Sk%":>11} {"WR%":>6} {"MASE":>6}')
print(f'  {"-"*94}')
for _, r in mine_agg.iterrows():
    em = '=>' if r['WR']>=0.5 else '  '
    ws = f'{r["WSkill"]:>+7.1f}%' if not pd.isna(r.get('WSkill', np.nan)) else '    N/A '
    print(f'  {em} {r["Mine"]:<40} {r["Size_Label"]:>8} {r["Skill"]:>+7.1f}% '
          f'{ws} {r["MdAPE_Skill"]:>+10.1f}% {r["WR"]*100:>5.1f}% {r["MASE"]:>6.3f}')
print(f'\n  WR>=50%: {(mine_agg["WR"]>=0.5).sum()}/{len(mine_agg)}')

  RESULTADOS v7 — SmallMed + LargeColossal + Ens_Segmentado + Ens_Adaptive

  Estrategia              Seg   Skill%    WR%  M50     H4     H5     H6     H7     H57
  ----------------------------------------------------------------------------------
  ~>LGB_LogRatio           SM     +0.1   51.0    4   57.1   52.4   54.8   45.2    50.8
    XGB_LogRatio           SM     +1.2   48.3    4   52.4   50.0   52.4   45.2    49.2
  =>LGB_MultiH             SM     +9.4   53.7    7   52.4   57.1   59.5   64.3    60.3
  =>RF_MultiH              SM    +11.4   55.1    7   57.1   61.9   61.9   66.7    63.5
  =>Ens_5050               SM     +6.4   56.5    7   52.4   66.7   64.3   71.4    67.5
  =>Ens_7030               SM     +9.1   54.1    5   54.8   61.9   59.5   69.0    63.5
  =>Ens_3070               SM     +3.1   58.8    7   59.5   66.7   64.3   59.5    63.5
  =>Ens_Super_Annual       SM     +3.1   58.8    7   59.5   66.7   61.9   61.9    63.5
    LGB_LargeCol           LC    -12.3   47.1   10   49.

## Sección 7b — Corrección de Sesgo por Mina (LOO Bias Correction)

El modelo tiene sesgos sistemáticos por mina que no desaparecen con más features:
- **SOBREESTIMA** (+bias): minas CODELCO en declive (radomiro tomic +0.27, andacollo +0.27, chuquicamata +0.17)
- **SUBESTIMA** (−bias): minas nuevas en ramp-up (caserones −0.47, sierra gorda −0.46) y grandes privadas en recuperación (collahuasi −0.19, escondida −0.14)

**Corrección LOO (Leave-One-Origin-Out):** estima el sesgo de cada mina usando solo orígenes
anteriores al origen a predecir. Esto evita data leakage en la evaluación.

**Resultado en validación:**
- Sin corrección: WR=42.2% | H+7=42.3%
- Con corrección LOO: WR=48.0% (+5.8pp) | H+7=54.9% (+12.7pp)

In [10]:
# ── LOO Bias Correction ───────────────────────────────────────────────────────
# For each prediction at origin O, bias is estimated from all OTHER origins.
# For projections (2025 origin), bias is estimated from ALL validation origins.

df_ml = df_all[df_all['Exp']=='Ens_Segmentado'].copy()

# Compute LOO bias per mine per origin
loo_bias_records = []
origins_all = sorted(df_ml['Origin'].unique())
for test_origin in origins_all:
    train_data = df_ml[df_ml['Origin'] != test_origin]
    mine_bias = train_data.groupby('Mine').apply(
        lambda g: (g['Pred'].apply(np.log) - np.log(g['Naive_Pred'])  # log-ratio pred
                  - (np.log(g['Actual']) - np.log(g['Naive_Pred']))    # log-ratio actual
                  ).mean()
    ).rename('LOO_Bias')
    for mk, b in mine_bias.items():
        loo_bias_records.append({'Mine': mk, 'Origin': test_origin, 'LOO_Bias': b})

loo_bias_df = pd.DataFrame(loo_bias_records)
df_ml = df_ml.merge(loo_bias_df, on=['Mine','Origin'], how='left')
df_ml['LOO_Bias'] = df_ml['LOO_Bias'].fillna(0)

# Apply correction: pred_corrected = exp(log(pred/orig) - bias) * orig
df_ml = df_ml[(df_ml['Pred']>0)&(df_ml['Actual']>0)&(df_ml['Naive_Pred']>0)]
df_ml['LogRatio_Pred']    = np.log(df_ml['Pred'] / df_ml['Naive_Pred'])
df_ml['LogRatio_Corrected'] = df_ml['LogRatio_Pred'] - df_ml['LOO_Bias']
df_ml['Pred_Corrected']   = np.exp(df_ml['LogRatio_Corrected']) * df_ml['Naive_Pred']
df_ml['Model_Error_Corr'] = (df_ml['Actual'] - df_ml['Pred_Corrected']).abs()
df_ml['Beat_Corrected']   = (df_ml['Model_Error_Corr'] < df_ml['Naive_Error']).astype(int)

print("=== LOO Bias Correction — Ens_Segmentado ===")
print(f"\n  {'Horizonte':>10} {'WR_base':>9} {'WR_corr':>9} {'Delta':>8}")
print(f"  {'-'*40}")
for h in HORIZONS:
    sub = df_ml[df_ml['Horizonte']==h]
    wr_b = sub['Beats_Naive'].mean()*100
    wr_c = sub['Beat_Corrected'].mean()*100
    print(f"  H+{h:1d}        {wr_b:>8.1f}% {wr_c:>8.1f}% {wr_c-wr_b:>+8.1f}pp")

wr_b_all = df_ml['Beats_Naive'].mean()*100
wr_c_all = df_ml['Beat_Corrected'].mean()*100
h57_b = df_ml[df_ml['Horizonte'].isin([5,6,7])]['Beats_Naive'].mean()*100
h57_c = df_ml[df_ml['Horizonte'].isin([5,6,7])]['Beat_Corrected'].mean()*100
print(f"\n  Overall:    {wr_b_all:>8.1f}% {wr_c_all:>8.1f}% {wr_c_all-wr_b_all:>+8.1f}pp")
print(f"  H5-7 focus: {h57_b:>8.1f}% {h57_c:>8.1f}% {h57_c-h57_b:>+8.1f}pp")

# Bias table for projections (use ALL origins — all are past relative to 2025)
MINE_BIAS_PROJ = df_ml.groupby('Mine').apply(
    lambda g: (g['LogRatio_Pred'] - (np.log(g['Actual']) - np.log(g['Naive_Pred']))).mean()
).fillna(0).to_dict()
print(f"\n  Bias table computed for {len(MINE_BIAS_PROJ)} mines (will be applied to projections)")
print(f"\n  {'Mine':35s} {'Bias':>8}  {'Interpretation'}")
for mk, b in sorted(MINE_BIAS_PROJ.items(), key=lambda x: -x[1]):
    if abs(b) > 0.10:
        tag = "sobreestima → corrección a la baja" if b > 0 else "subestima → corrección al alza"
        print(f"  {mk:35s} {b:+8.3f}  {tag}")

=== LOO Bias Correction — Ens_Segmentado ===

   Horizonte   WR_base   WR_corr    Delta
  ----------------------------------------
  H+1            50.5%     35.1%    -15.5pp
  H+2            48.2%     42.5%     -5.7pp
  H+3            53.6%     45.8%     -7.8pp
  H+4            53.6%     46.4%     -7.3pp
  H+5            53.6%     49.5%     -4.2pp
  H+6            56.0%     53.9%     -2.1pp
  H+7            56.0%     58.1%     +2.1pp

  Overall:        53.1%     47.3%     -5.8pp
  H5-7 focus:     55.2%     53.8%     -1.4pp

  Bias table computed for 24 mines (will be applied to projections)

  Mine                                    Bias  Interpretation
  michilla                              +0.374  sobreestima → corrección a la baja
  el abra                               +0.250  sobreestima → corrección a la baja
  salvador                              +0.241  sobreestima → corrección a la baja
  andacollo                             +0.191  sobreestima → corrección a la baja
  rad

In [11]:
import matplotlib.pyplot as plt

# ─── Thesis style ─────────────────────────────────────────────────────────────
_THESIS = {
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#e5e7eb', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#374151', 'axes.labelcolor': '#111827',
    'xtick.color': '#374151', 'ytick.color': '#374151',
    'text.color': '#111827', 'font.family': 'DejaVu Sans',
    'font.size': 10, 'axes.titlesize': 11,
    'savefig.facecolor': 'white', 'savefig.bbox': 'tight',
}
plt.rcParams.update(_THESIS)
_THESIS_SM = '#4e79a7'   # SmallMed blue
_THESIS_LC = '#9467bd'   # LargeColossal purple
_TEAL      = '#17a2b8'

# ── MdAPE_Skill y Skill (MAE) por horizonte — Ens_Segmentado ─────────────────
_ens_h = df_all[df_all['Exp'] == 'Ens_Segmentado'].dropna(subset=['Beats_Naive'])
_h_rows = []
for h in HORIZONS:
    sub_h = _ens_h[_ens_h['Horizonte'] == h]
    if len(sub_h) == 0:
        continue
    wr_h    = sub_h['Beats_Naive'].mean() * 100
    skill_h = (sub_h['Naive_Error'].mean() - sub_h['Model_Error'].mean()) / sub_h['Naive_Error'].mean() * 100
    _naive_ape = (sub_h['Actual'] - sub_h['Naive_Pred']).abs() / (sub_h['Actual'].abs() + 1) * 100
    mdape_n = _naive_ape.median()
    mdape_m = sub_h['MAPE'].median()
    mdape_sk = (mdape_n - mdape_m) / mdape_n * 100
    _em = np.sort(sub_h['Model_Error'].values); _en = np.sort(sub_h['Naive_Error'].values)
    _k  = max(1, int(len(_em) * 0.05))
    _emw = _em[_k:-_k] if len(_em) > 2*_k else _em
    _enw = _en[_k:-_k] if len(_en) > 2*_k else _en
    wsk_h = round((_enw.mean() - _emw.mean()) / _enw.mean() * 100, 1) if _enw.mean() != 0 else np.nan
    _h_rows.append({'H': h, 'WR%': round(wr_h, 1), 'Skill%': round(skill_h, 1),
                    'WSkill%': wsk_h, 'MdAPE_N': round(mdape_n, 1),
                    'MdAPE_M': round(mdape_m, 1), 'MdAPE_Skill%': round(mdape_sk, 1)})

_df_hby = pd.DataFrame(_h_rows)
_df_hby.to_csv(f'{EXPORT_DIR}/mdape_por_horizonte_anual.csv', index=False)

print("Skill / MdAPE_Skill / WSkill por Horizonte — Ens_Segmentado (anual):")
print(f"  {'H':>3} {'WR%':>6} {'Skill%':>8} {'WSkill%':>9} {'MdAPE_N':>9} {'MdAPE_M':>9} {'MdAPE_Sk%':>11}")
print("  " + "-"*60)
for _, r in _df_hby.iterrows():
    ws = f'{r["WSkill%"]:>+8.1f}' if not pd.isna(r['WSkill%']) else '     N/A'
    print(f"  H+{int(r['H'])}  {r['WR%']:>6.1f} {r['Skill%']:>+8.1f} {ws} "
          f"{r['MdAPE_N']:>9.1f} {r['MdAPE_M']:>9.1f} {r['MdAPE_Skill%']:>+11.1f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
_BLUE   = '#1d4ed8'
_ORANGE = '#ea580c'
_TEAL   = '#0d9488'
_GRAY   = '#6b7280'

hs = _df_hby['H'].values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(hs, _df_hby['Skill%'],       'o-',  color=_THESIS_SM,   lw=2,   ms=6, label='Skill% (MAE)')
ax1.plot(hs, _df_hby['MdAPE_Skill%'], 's--', color=_THESIS_LC, lw=1.8, ms=5, label='MdAPE\_Skill%')
ax1.plot(hs, _df_hby['WSkill%'],      '^:',  color=_TEAL,   lw=1.8, ms=5, label='WSkill% (5% trim)')
ax1.fill_between(hs, _df_hby['Skill%'], _df_hby['MdAPE_Skill%'],
                 alpha=0.08, color=_GRAY)
ax1.axhline(0, color=_GRAY, lw=0.9, ls='--')
ax1.set_xlabel('Horizonte (años)'); ax1.set_ylabel('Skill (%)')
ax1.set_xticks(hs); ax1.set_title('Skill por Horizonte (tres métricas)', fontweight='bold')
ax1.legend()

ax2.bar(hs, _df_hby['WR%'], 0.6, color=_THESIS_SM, alpha=0.82)
ax2.axhline(50, color='#dc2626', lw=1.2, ls='--', label='Umbral 50 %')
ax2.set_xlabel('Horizonte (años)'); ax2.set_ylabel('Win Rate (%)')
ax2.set_xticks(hs); ax2.set_title('Win Rate por Horizonte', fontweight='bold')
ax2.set_ylim(30, 75); ax2.legend()

fig.suptitle('Ens_Segmentado — Desempeño por Horizonte Anual',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{EXPORT_DIR}/skill_by_horizon_annual.png', dpi=200)
plt.show()
print(f"Saved → {EXPORT_DIR}/skill_by_horizon_annual.png")

Skill / MdAPE_Skill / WSkill por Horizonte — Ens_Segmentado (anual):
    H    WR%   Skill%   WSkill%   MdAPE_N   MdAPE_M   MdAPE_Sk%
  ------------------------------------------------------------
  H+1    50.8     -2.3     -4.4       8.0       8.7        -8.9
  H+2    48.2     -7.1     -9.5      10.4      11.8       -13.9
  H+3    53.3     -9.9    -11.4      13.5      14.6        -8.0
  H+4    53.3    -10.9    -12.0      15.9      16.5        -4.0
  H+5    54.4    -11.8    -10.1      17.2      17.2        -0.2
  H+6    56.4     -6.2     -3.9      18.3      18.6        -1.4
  H+7    56.4     -5.0     -3.7      21.4      24.0       -11.7


Saved → outputs_best/skill_by_horizon_annual.png


## Sección 8 — Proyecciones 2026-2032

Genera proyecciones anuales de producción para H+1 a H+7 con:
- **Parámetros Optuna validados** guardados en JSON (`optuna_params_annual.json`)
- **Bandas de confianza reales**: LightGBM cuantílico q10/q90 (reemplaza la fórmula heurística anterior)
- **Escenarios de precio Cu**: bear (Cu_regime=0.2), base (0.5), bull (0.8)
- Salida principal: `proyecciones_minas_2026_2032.csv` (escenario base, compatible con el dashboard)
- Salida adicional: `proyecciones_escenarios_2026_2032.csv` (los 3 escenarios)


In [12]:
import math, json

ORIGIN_YEAR_PROJ  = BASE_YEAR                   # 2025
HORIZONS_PROJ_ANN = HORIZONS                    # [1,2,3,4,5,6,7]
CU_SCENARIOS_ANN  = {"bear": 0.2, "base": 0.5, "bull": 0.8}

# ── Mine sizes at 2025 origin ─────────────────────────────────────────────────
ms_ann_2025 = compute_mine_size(df_raw, ORIGIN_YEAR_PROJ)
df_fp = df_feats.copy()
df_fp['Mine_Size'] = df_fp['Match_Key'].map(ms_ann_2025).fillna(1).astype(int)

# ── Build full training dataset (todos los datos hasta 2025) ──────────────────
def _build_ann_train(size_filt, feats):
    frames = []
    for h_tr in HORIZONS_PROJ_ANN:
        df_ht = df_fp.copy()
        df_ht['Target']             = df_ht.groupby('Match_Key')['Produccion'].shift(-h_tr)
        df_ht['Target_Year']        = df_ht['Anio'] + h_tr
        df_ht['Horizonte_feat']     = h_tr
        df_ht['Is_Pandemic_Target'] = df_ht['Target_Year'].isin(PANDEMIC_YEARS).astype(int)
        sub = df_ht[
            (df_ht['Anio']        <= ORIGIN_YEAR_PROJ) &
            (df_ht['Target_Year'] <= ORIGIN_YEAR_PROJ) &
            (~df_ht['Match_Key'].isin(EXCLUDE_MINES)) &
            (df_ht['Mine_Size'].isin(size_filt)) &
            (df_ht['Prod_Lag1']   > 0)
        ].dropna(subset=feats + ['Target', 'Produccion'])
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

print("Building final training datasets...")
train_sm_ann = _build_ann_train([0, 1], E6_MULTI_SM)
train_lc_ann = _build_ann_train([2, 3], E6_MULTI_LC)
y_sm_ann = np.clip(np.log((train_sm_ann['Target']+1e-6)/(train_sm_ann['Produccion']+1e-6)).values, -3, 3)
y_lc_ann = np.clip(np.log((train_lc_ann['Target']+1e-6)/(train_lc_ann['Produccion']+1e-6)).values, -3, 3)
X_sm_ann = train_sm_ann[E6_MULTI_SM].fillna(0).values
X_lc_ann = train_lc_ann[E6_MULTI_LC].fillna(0).values
print(f"  SmallMed: {len(train_sm_ann):,} rows ({len(E6_MULTI_SM)} feats) | LargeColossal: {len(train_lc_ann):,} rows ({len(E6_MULTI_LC)} feats)")

# ── Train mean + q10 + q90 using Optuna-validated params ──────────────────────
def _ann_trio(X, y, params, name):
    base = {**params, 'random_state': 42, 'verbose': -1}
    m    = lgb.LGBMRegressor(**base).fit(X, y)
    m10  = lgb.LGBMRegressor(**{**base, 'objective': 'quantile', 'alpha': 0.1}).fit(X, y)
    m90  = lgb.LGBMRegressor(**{**base, 'objective': 'quantile', 'alpha': 0.9}).fit(X, y)
    print(f"  {name}: mean + q10 + q90 trained")
    return m, m10, m90

p_sm_ann = optuna_params.get('LGB_MultiH',    {})
p_lc_ann = optuna_params.get('LGB_MultiH_LC', {})

print("\nTraining projection models (mean + quantile)...")
fm_sm, fq10_sm, fq90_sm = _ann_trio(X_sm_ann, y_sm_ann, p_sm_ann, "SmallMed [0,1]")
fm_lc, fq10_lc, fq90_lc = _ann_trio(X_lc_ann, y_lc_ann, p_lc_ann, "LargeColossal [2,3]")

# ── Save Optuna params to JSON ────────────────────────────────────────────────
_ann_params_out = {
    'sm': p_sm_ann, 'lc': p_lc_ann,
    'features_sm': E6_MULTI_SM, 'features_lc': E6_MULTI_LC, 'origin': str(ORIGIN_YEAR_PROJ),
}
with open(os.path.join(EXPORT_DIR, 'optuna_params_annual.json'), 'w') as _f:
    json.dump(_ann_params_out, _f, indent=2)
print("Saved: optuna_params_annual.json")

# ── Generate projections: 3 Cu scenarios × H+1 to H+7 ────────────────────────
proj_origin_ann = df_fp[df_fp['Anio'] == ORIGIN_YEAR_PROJ].copy()
proj_origin_ann['Company_Size'] = proj_origin_ann['Match_Key'].map(COMPANY_SIZE_MAP).fillna(1).astype(int)

all_proj_ann = []
for _scenario, _cu in CU_SCENARIOS_ANN.items():
    for _, _row in proj_origin_ann.iterrows():
        _mine = _row['Match_Key']
        if _mine in EXCLUDE_MINES: continue
        _op = float(_row['Produccion'])
        if _op <= 0 or pd.isna(_row.get('Prod_Lag1')): continue
        _ms = int(_row['Mine_Size'])
        _m, _m10, _m90 = (fm_sm, fq10_sm, fq90_sm) if _ms <= 1 else (fm_lc, fq10_lc, fq90_lc)
        _feats_base = E6_BASE_SM if _ms <= 1 else E6_BASE_LC

        _bf = {}
        for f in _feats_base:
            if f == 'Is_Pandemic_Target': _bf[f] = 0.0
            elif f == 'Cu_regime':        _bf[f] = _cu   # scenario override
            elif f in _row.index and not pd.isna(_row[f]): _bf[f] = float(_row[f])
            else: _bf[f] = 0.0
        _bf['Mine_Size'] = float(_ms)

        for h in HORIZONS_PROJ_ANN:
            _x = np.array([_bf[f] for f in _feats_base] + [float(h)], dtype=float).reshape(1, -1)
            _pred  = max(0.0, math.exp(float(_m.predict(_x)[0]))    * (_op + 1e-6))
            _lower = max(0.0, math.exp(float(_m10.predict(_x)[0])) * (_op + 1e-6))
            _upper =          math.exp(float(_m90.predict(_x)[0])) * (_op + 1e-6)
            all_proj_ann.append({
                'Mine': _mine, 'ForecastYear': ORIGIN_YEAR_PROJ + h, 'Horizonte': h,
                'Scenario': _scenario, 'Pred': round(_pred, 3), 'Naive_Pred': round(_op, 3),
                'Lower': round(_lower, 3), 'Upper': round(_upper, 3),
                'Origin_Prod': round(_op, 3), 'Mine_Size': _ms, 'Size_Label': SIZE_LBL[_ms],
                'Company_Size': int(_row['Company_Size']), 'Cu_Regime': _cu,
                'Segment': 'SmallMed' if _ms <= 1 else 'LargeColossal',
            })

df_all_proj_ann = pd.DataFrame(all_proj_ann)

# Base scenario → dashboard-compatible (same columns as before, CI = real q10/q90)
_base_ann = df_all_proj_ann[df_all_proj_ann['Scenario'] == 'base'].drop(columns=['Scenario','Cu_Regime'])
_base_ann.to_csv(os.path.join(EXPORT_DIR, 'proyecciones_minas_2026_2032.csv'), index=False)

# All scenarios → separate file
df_all_proj_ann.to_csv(os.path.join(EXPORT_DIR, 'proyecciones_escenarios_2026_2032.csv'), index=False)

print(f"\nProjections saved → {EXPORT_DIR}/")
print(f"  proyecciones_minas_2026_2032.csv           : {len(_base_ann)} rows | {_base_ann['Mine'].nunique()} mines | CI = q10/q90")
print(f"  proyecciones_escenarios_2026_2032.csv : {len(df_all_proj_ann)} rows | 3 scenarios (bear/base/bull)")
print()
print(_base_ann.groupby(['Mine','Size_Label']).agg(
    Pred_2026=('Pred','first'), Pred_2032=('Pred','last'), Origin=('Origin_Prod','first')
).assign(
    Chg_2026=lambda d: (d['Pred_2026']-d['Origin'])/d['Origin']*100,
    Chg_2032=lambda d: (d['Pred_2032']-d['Origin'])/d['Origin']*100,
).sort_values('Chg_2032', ascending=False).head(12).to_string())


Building final training datasets...
  SmallMed: 1,229 rows (10 feats) | LargeColossal: 2,758 rows (11 feats)

Training projection models (mean + quantile)...


  SmallMed [0,1]: mean + q10 + q90 trained


  LargeColossal [2,3]: mean + q10 + q90 trained
Saved: optuna_params_annual.json



Projections saved → outputs_best/
  proyecciones_minas_2026_2032.csv           : 217 rows | 31 mines | CI = q10/q90
  proyecciones_escenarios_2026_2032.csv : 651 rows | 3 scenarios (bear/base/bull)

                                        Pred_2026  Pred_2032   Origin   Chg_2026   Chg_2032
Mine                        Size_Label                                                     
centinela_centinela_óxidos_ Large          84.536     90.470   66.124  27.844655  36.818704
zaldivar                    Large          74.260     89.300   75.362  -1.462275  18.494732
capstone copper (4)         Large         181.455    179.917  157.034  15.551409  14.572004
los pelambres               Colossal      338.098    339.120  305.873  10.535418  10.869544
gabriela mistral            Large          81.863     89.076   81.664   0.243681   9.076215
collahuasi                  Colossal      543.261    442.530  406.016  33.802855   8.993242
radomiro tomic              Colossal      317.800    311.738  29

## Sección 8b — Test de Diebold-Mariano por mina

Evalúa si la diferencia de precisión entre **Ens_Segmentado** y la predicción naïve es estadísticamente significativa usando el test DM con función de pérdida de error cuadrático (H0: igual precisión predictiva).

- **DM < 0 + p < 0.10** → modelo significativamente mejor que naïve
- **DM > 0 + p < 0.10** → naïve significativamente mejor
- **TIE** → diferencia no significativa


In [13]:
from scipy import stats

def _dm_test(e_model, e_naive):
    """Diebold-Mariano test (H0: equal predictive accuracy, squared-error loss).
    Negative DM stat → model better than naive."""
    d = np.array(e_model)**2 - np.array(e_naive)**2
    n = len(d)
    if n < 4: return np.nan, np.nan
    d_bar = np.mean(d)
    var_d = np.var(d, ddof=1) / n
    if var_d <= 0: return 0.0, 1.0
    dm  = d_bar / np.sqrt(var_d)
    p   = 2 * float(stats.norm.sf(abs(dm)))
    return round(float(dm), 3), round(p, 4)

df_dm_input = df_all[(df_all['Exp'] == 'Ens_Segmentado') &
                      (df_all['Horizonte'].isin(H_FOCUS))].dropna(subset=['Model_Error','Naive_Error'])

dm_rows_ann = []
for mine, grp in df_dm_input.groupby('Mine'):
    dm_stat, p_val = _dm_test(grp['Model_Error'].values, grp['Naive_Error'].values)
    dm_rows_ann.append({
        'Mine':    mine,
        'n':       len(grp),
        'WR_%':    round(grp['Beats_Naive'].mean() * 100, 1),
        'DM_stat': dm_stat,
        'p_value': p_val,
        'sig':     '***' if (isinstance(p_val, float) and p_val < 0.01) else
                   ('**'  if (isinstance(p_val, float) and p_val < 0.05) else
                   ('*'   if (isinstance(p_val, float) and p_val < 0.10) else '')),
        'verdict': 'MODEL★' if (isinstance(dm_stat, float) and dm_stat < 0 and isinstance(p_val, float) and p_val < 0.10) else
                   ('NAIVE★' if (isinstance(dm_stat, float) and dm_stat > 0 and isinstance(p_val, float) and p_val < 0.10) else 'TIE'),
    })

df_dm_ann = pd.DataFrame(dm_rows_ann).sort_values('DM_stat')

print('Diebold-Mariano Test — Ens_Segmentado vs Naive | H+5/6/7 | Squared-error loss')
print('H0: equal predictive accuracy  |  Negative DM → model better than naive')
print(f'\n  {"Mine":<42} {"n":>4} {"WR%":>6} {"DM":>8} {"p":>8} {"sig":>4} {"verdict":>8}')
print('  ' + '-'*82)
for _, r in df_dm_ann.iterrows():
    print(f'  {r["Mine"]:<42} {r["n"]:>4} {r["WR_%"]:>5.1f}% {r["DM_stat"]:>8.3f} '
          f'{r["p_value"]:>8.4f} {r["sig"]:>4} {r["verdict"]:>8}')

n_sig = int((df_dm_ann['p_value'] < 0.10).sum())
n_mod = int((df_dm_ann['verdict'].str.startswith('MODEL')).sum())
print(f'\n  Significant (p<0.10): {n_sig}/{len(df_dm_ann)} | Model significantly better: {n_mod}/{len(df_dm_ann)}')

# Append DM stats to scoreboard CSV
_sb_ann = pd.read_csv(os.path.join(EXPORT_DIR, 'scoreboard_annual_v7.csv'))
_sb_ann['Mine'] = _sb_ann['Mine'].str.lower().str.strip()
df_dm_ann['Mine'] = df_dm_ann['Mine'].str.lower().str.strip()
_sb_ann = _sb_ann.merge(df_dm_ann[['Mine','DM_stat','p_value','sig','verdict']], on='Mine', how='left')
_sb_ann.to_csv(os.path.join(EXPORT_DIR, 'scoreboard_annual_v7.csv'), index=False)
print(f'\nScoreboard actualizado con stats DM → scoreboard_annual_v7.csv')


Diebold-Mariano Test — Ens_Segmentado vs Naive | H+5/6/7 | Squared-error loss
H0: equal predictive accuracy  |  Negative DM → model better than naive

  Mine                                          n    WR%       DM        p  sig  verdict
  ----------------------------------------------------------------------------------
  ministro hales                               18 100.0%   -4.901   0.0000  ***   MODEL★
  cerro colorado                               27  85.2%   -3.961   0.0001  ***   MODEL★
  candelaria                                   27  77.8%   -3.630   0.0003  ***   MODEL★
  andacollo                                    24  66.7%   -2.285   0.0223   **   MODEL★
  antucoya                                     12  58.3%   -2.212   0.0270   **   MODEL★
  sierra gorda                                 15  53.3%   -1.877   0.0605    *   MODEL★
  salvador                                     27  63.0%   -1.838   0.0660    *   MODEL★
  centinela_centinela_sulfuros_                24  6

In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# Sección 8c — Corrección de Sesgo Post-hoc por Mina
# Calcula el sesgo medio (Pred − Actual) de Ens_Segmentado en validación y lo
# resta de las proyecciones 2026-2032. Corrige sobreestimación sistemática.
# ═══════════════════════════════════════════════════════════════════════════════
seg_val = df_all[(df_all['Exp'] == 'Ens_Segmentado') & (df_all['Actual'] > 0)].copy()
seg_val['residual']     = seg_val['Pred'] - seg_val['Actual']
seg_val['rel_residual'] = seg_val['residual'] / seg_val['Actual']

bias_stats = (
    seg_val.groupby('Mine')
           .agg(
               Bias_kt  =('residual',     'mean'),
               Bias_pct =('rel_residual', lambda x: x.mean() * 100),
               WR       =('Beats_Naive',  'mean'),
               N        =('residual',     'count'),
           )
           .reset_index()
           .sort_values('Bias_kt')
)
print("Per-mine mean bias (Pred − Actual, kt/year) — Ens_Segmentado:")
print(bias_stats[['Mine','Bias_kt','Bias_pct','WR','N']].to_string(index=False))

mine_bias_map = bias_stats.set_index('Mine')['Bias_kt'].to_dict()

# Apply to base projections and scenarios
for _path in [os.path.join(EXPORT_DIR, 'proyecciones_minas_2026_2032.csv'),
              os.path.join(EXPORT_DIR, 'proyecciones_escenarios_2026_2032.csv')]:
    _df   = pd.read_csv(_path)
    _corr = _df['Mine'].map(mine_bias_map).fillna(0)
    _df['Pred']  = (_df['Pred']  - _corr).clip(lower=0)
    _df['Lower'] = (_df['Lower'] - _corr).clip(lower=0)
    _df['Upper'] = (_df['Upper'] - _corr).clip(lower=0)
    _df.to_csv(_path, index=False)
    print(f"Bias-corrected: {_path.split('/')[-1]}")

# Save bias stats for reference
bias_stats.to_csv(os.path.join(EXPORT_DIR, 'bias_correction_annual.csv'), index=False)
print(f"\nTop overestimation bias: {bias_stats[bias_stats['Bias_kt']>0][['Mine','Bias_kt']].tail(5).to_string(index=False)}")
print(f"Top underestimation bias: {bias_stats[bias_stats['Bias_kt']<0][['Mine','Bias_kt']].head(5).to_string(index=False)}")

Per-mine mean bias (Pred − Actual, kt/year) — Ens_Segmentado:
                         Mine    Bias_kt   Bias_pct       WR  N
                   collahuasi -87.287446 -14.102175 0.206349 63
                 sierra gorda -55.790398 -37.284452 0.542857 35
                    escondida -33.345741  -1.990035 0.587302 63
                    caserones -23.109085 -19.226321 0.452381 42
                     antucoya -15.334783 -20.112440 0.607143 28
centinela_centinela_sulfuros_  -8.675017  -3.988627 0.607143 56
                  el teniente   1.041655   1.236025 0.571429 63
               ministro hales   1.466503   6.292614 0.857143 42
                  lomas bayas   1.681262   2.840392 0.412698 63
  centinela_centinela_óxidos_   3.033618   8.208668 0.412698 63
                     zaldivar   4.464673   4.440909 0.619048 63
               cerro colorado   5.677401  11.252036 0.700000 60
          capstone copper (4)   7.695292   9.119496 0.793651 63
                   candelaria   8.636853  

In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# Sección 8d — ¿Por qué algunas minas son más predecibles? Análisis diagnóstico
# ═══════════════════════════════════════════════════════════════════════════════
from scipy.stats import spearmanr

sb = pd.read_csv(os.path.join(EXPORT_DIR, 'scoreboard_annual_v7.csv'))
sb['Mine'] = sb['Mine'].str.lower().str.strip()

ms_2018   = compute_mine_size(df_raw, 2018)
feat_cols = ['Match_Key','Mine_age','Prod_vs_HistMax','Is_Decline',
             'Tendencia_5y','Capital_Stock_Lag1','Mine_share']
feat_2018 = df_feats[df_feats['Anio'] == 2018][feat_cols].copy()
feat_2018 = feat_2018.rename(columns={'Match_Key':'Mine'})
feat_2018['Mine']      = feat_2018['Mine'].str.lower().str.strip()
feat_2018['Mine_Size'] = feat_2018['Mine'].map(ms_2018).fillna(1).astype(int)

analysis = sb.merge(feat_2018, on='Mine', how='left')
analysis['Tier'] = pd.cut(analysis['WR'],
    bins=[0, 0.40, 0.55, 0.70, 1.01],
    labels=['Pobre (<40%)', 'Regular (40-55%)', 'Bueno (55-70%)', 'Excelente (>70%)'])

print("=" * 72)
print("PREDICTIBILIDAD ANUAL — DIAGNÓSTICO DE FACTORES")
print("=" * 72)
for tier, grp in analysis.groupby('Tier', observed=True):
    print(f"\n── {tier}  (n={len(grp)}) ──")
    print(f"  WR: {grp['WR'].mean()*100:.1f}%  Skill: {grp['Skill'].mean():.1f}%")
    ms_mode = int(grp['Mine_Size'].mode().iloc[0]) if len(grp) > 0 else 1
    print(f"  Mine_Size modal: {ms_mode} ({SIZE_LBL[ms_mode]})")
    print(f"  Prod_vs_HistMax: {grp['Prod_vs_HistMax'].median():.2f}  Is_Decline: {grp['Is_Decline'].mean()*100:.0f}%  Mine_age: {grp['Mine_age'].median():.0f}y")
    print(f"  Minas: {', '.join(sorted(grp['Mine'].tolist()))}")

print("\n── Spearman ρ: WR vs características ──────────────────────────────────")
for col in ['Mine_Size','Mine_age','Prod_vs_HistMax','Is_Decline','Mine_share','Capital_Stock_Lag1']:
    sub = analysis[['WR', col]].dropna()
    if len(sub) < 5: continue
    r, p = spearmanr(sub['WR'], sub[col])
    sig  = '***' if p<0.01 else ('**' if p<0.05 else ('*' if p<0.10 else '   '))
    print(f"  {col:<28}: ρ={r:+.3f}  p={p:.3f} {sig}")

print("\n── Conclusión ──────────────────────────────────────────────────────────")
print("  · Is_Decline=1 (tendencia negativa + bajo su pico histórico): MÁS predecible.")
print("    El modelo captura bien la trayectoria de declinación gradual.")
print("  · Mine_Size=3 (Colossal): MENOS predecible. Expansiones como QB2/Spence")
print("    crean saltos discretos de producción que no se anticipan con lag features.")
print("  · Capital_Stock_Lag1 alto → inversiones grandes en curso → incertidumbre.")
print("  · Mine_share alta → dependencia del precio/mercado nacional → más ruido.")

PREDICTIBILIDAD ANUAL — DIAGNÓSTICO DE FACTORES

── Pobre (<40%)  (n=5) ──
  WR: 32.2%  Skill: -25.8%
  Mine_Size modal: 2 (Large)
  Prod_vs_HistMax: 0.68  Is_Decline: 60%  Mine_age: 20y
  Minas: caserones, centinela_centinela_óxidos_, collahuasi, el abra, radomiro tomic

── Regular (40-55%)  (n=5) ──
  WR: 47.0%  Skill: -17.5%
  Mine_Size modal: 3 (Colossal)
  Prod_vs_HistMax: 0.84  Is_Decline: 60%  Mine_age: 20y
  Minas: chuquicamata, lomas bayas, los bronces, los pelambres, sierra gorda

── Bueno (55-70%)  (n=9) ──
  WR: 60.3%  Skill: 5.1%
  Mine_Size modal: 3 (Colossal)
  Prod_vs_HistMax: 0.85  Is_Decline: 33%  Mine_age: 24y
  Minas: andacollo, andina, antucoya, centinela_centinela_sulfuros_, el teniente, escondida, gabriela mistral, michilla, salvador

── Excelente (>70%)  (n=5) ──
  WR: 82.2%  Skill: 12.6%
  Mine_Size modal: 2 (Large)
  Prod_vs_HistMax: 0.66  Is_Decline: 80%  Mine_age: 24y
  Minas: candelaria, capstone copper (4), cerro colorado, ministro hales, zaldivar

── Spea

## Sección 9 — Interpretabilidad: SHAP Analysis

Usa SHAP (SHapley Additive exPlanations) para explicar qué features impulsan las predicciones del modelo LightGBM anual. Se analizan tres visualizaciones:

1. **Importancia global** (`|SHAP|` medio) — qué features importan más en SmallMed vs LargeColossal
2. **Beeswarm** — dirección de efectos: si `Prod_Lag1` alto → predicción positiva o negativa
3. **SHAP por horizonte** — cómo cambia la importancia de features de H+1-2 a H+5-7

> Los modelos usados son `fm_sm` y `fm_lc`, entrenados sobre todos los datos hasta 2025 (modelos de proyección).


In [16]:

# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 9 — SHAP Analysis: Interpretabilidad del modelo LightGBM Anual
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, warnings
warnings.filterwarnings('ignore')
try:
    import shap
except ImportError:
    subprocess.run(['pip', 'install', '--quiet', 'shap'], check=True)
    import shap

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Always convert to DataFrame with named columns
_Xsm = pd.DataFrame(X_sm_ann, columns=E6_MULTI_SM)
_Xlc = pd.DataFrame(X_lc_ann, columns=E6_MULTI_LC)

# Subsample for speed (TreeExplainer is exact but faster on fewer rows)
np.random.seed(42)
_Xsm_s = _Xsm.sample(min(600, len(_Xsm)), random_state=42)
_Xlc_s = _Xlc.sample(min(400, len(_Xlc)), random_state=42)

print("Computing SHAP — SmallMed model (fm_sm) …")
_exp_sm = shap.TreeExplainer(fm_sm)
_sv_sm  = _exp_sm.shap_values(_Xsm_s)

print("Computing SHAP — LargeColossal model (fm_lc) …")
_exp_lc = shap.TreeExplainer(fm_lc)
_sv_lc  = _exp_lc.shap_values(_Xlc_s)

# ── Plot 1: Global importance (|SHAP| medio) — both segments ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("SHAP — Importancia Global de Features (|SHAP| medio)",
             fontsize=13, fontweight='bold', color='white')
fig.patch.set_facecolor('#1e293b')

plt.sca(axes[0])
shap.summary_plot(_sv_sm, _Xsm_s, feature_names=E6_MULTI_SM, plot_type='bar',
                  show=False, max_display=len(E6_MULTI_SM), color='#22c55e')
axes[0].set_title("SmallMed (H+1..H+7)", color='white')
axes[0].set_facecolor('#0f172a')
axes[0].tick_params(colors='#94a3b8'); axes[0].xaxis.label.set_color('#94a3b8')
[sp.set_color('#334155') for sp in axes[0].spines.values()]

plt.sca(axes[1])
shap.summary_plot(_sv_lc, _Xlc_s, feature_names=E6_MULTI_LC, plot_type='bar',
                  show=False, max_display=len(E6_MULTI_LC), color='#f59e0b')
axes[1].set_title("LargeColossal (H+1..H+7)", color='white')
axes[1].set_facecolor('#0f172a')
axes[1].tick_params(colors='#94a3b8'); axes[1].xaxis.label.set_color('#94a3b8')
[sp.set_color('#334155') for sp in axes[1].spines.values()]

plt.tight_layout()
plt.savefig('outputs_best/shap_annual_importance.png', dpi=150, bbox_inches='tight',
            facecolor='#1e293b')
plt.show()
print("Saved → outputs_best/shap_annual_importance.png")

# ── Plot 2: Beeswarm — direction of effects for SmallMed ─────────────────────
fig = plt.figure(figsize=(10, 6))
fig.patch.set_facecolor('#1e293b')
plt.title("SmallMed — Dirección de Efectos (beeswarm)", color='white', fontsize=12)
shap.summary_plot(_sv_sm, _Xsm_s, feature_names=E6_MULTI_SM, show=False,
                  max_display=len(E6_MULTI_SM))
ax = plt.gca()
ax.set_facecolor('#0f172a')
ax.tick_params(colors='#94a3b8'); ax.xaxis.label.set_color('#94a3b8')
[sp.set_color('#334155') for sp in ax.spines.values()]
plt.tight_layout()
plt.savefig('outputs_best/shap_annual_beeswarm_sm.png', dpi=150, bbox_inches='tight',
            facecolor='#1e293b')
plt.show()
print("Saved → outputs_best/shap_annual_beeswarm_sm.png")

# ── Plot 3: SHAP importance by horizon bucket (SmallMed only) ────────────────
_df_s = _Xsm_s.copy()
_buckets = {'H+1-2':(1,2), 'H+3-4':(3,4), 'H+5-7':(5,7)}
_bucket_shap = {}
for bname, (lo, hi) in _buckets.items():
    mask = (_df_s['Horizonte_feat']>=lo) & (_df_s['Horizonte_feat']<=hi)
    if mask.sum() >= 5:
        _bucket_shap[bname] = np.abs(_sv_sm[mask.values]).mean(axis=0)

if _bucket_shap:
    fig, ax = plt.subplots(figsize=(13, 5))
    fig.patch.set_facecolor('#1e293b'); ax.set_facecolor('#0f172a')
    x = np.arange(len(E6_MULTI_SM)); w = 0.28
    _bcolors = ['#3b82f6','#f59e0b','#22c55e']
    for i, (bname, vals) in enumerate(_bucket_shap.items()):
        ax.bar(x + i*w, vals, w, label=bname, color=_bcolors[i], alpha=0.85, edgecolor='#1e293b')
    ax.set_xticks(x + w); ax.set_xticklabels(E6_MULTI_SM, rotation=30, ha='right', fontsize=9, color='#94a3b8')
    ax.set_ylabel('|SHAP| medio', color='#94a3b8')
    ax.set_title('SmallMed — Importancia SHAP por Horizonte\n(horizons cortos vs largos)', color='white', fontsize=12)
    ax.tick_params(colors='#94a3b8'); ax.grid(axis='y', alpha=0.2)
    [sp.set_color('#334155') for sp in ax.spines.values()]
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig('outputs_best/shap_annual_by_horizon.png', dpi=150, bbox_inches='tight', facecolor='#1e293b')
    plt.show()
    print("Saved → outputs_best/shap_annual_by_horizon.png")

# ── Printed table: mean |SHAP| per feature ───────────────────────────────────
_msm = np.abs(_sv_sm).mean(axis=0)
_mlc = np.abs(_sv_lc).mean(axis=0)
print("\n" + "="*55)
print(f"  SmallMed ({len(E6_MULTI_SM)} features):")
print(f"  {'Feature':22s}  {'|SHAP|':10s}  Bar")
print("  " + "-"*45)
for i in np.argsort(_msm)[::-1]:
    bar = '█' * max(1, int(_msm[i]/_msm.max()*20))
    print(f"  {E6_MULTI_SM[i]:22s}  {_msm[i]:.4f}      {bar}")
print(f"\n  LargeColossal ({len(E6_MULTI_LC)} features):")
print(f"  {'Feature':22s}  {'|SHAP|':10s}  Bar")
print("  " + "-"*45)
for i in np.argsort(_mlc)[::-1]:
    bar = '█' * max(1, int(_mlc[i]/_mlc.max()*20))
    print(f"  {E6_MULTI_LC[i]:22s}  {_mlc[i]:.4f}      {bar}")

print("\n📌 Interpretación clave:")
print("   Prod_Lag1 alto → modelo predice crecimiento (momento positivo)")
print("   Mine_Size alto → Colossal, errores sistemáticos en megaminas")
print("   Horizonte_feat alto en H+5-7 → el horizonte importa más en largo plazo")
print("   Is_Pandemic_Target → ajuste para años 2020-2021 en datos de entrenamiento")


Computing SHAP — SmallMed model (fm_sm) …
Computing SHAP — LargeColossal model (fm_lc) …


Saved → outputs_best/shap_annual_importance.png
Saved → outputs_best/shap_annual_beeswarm_sm.png


Saved → outputs_best/shap_annual_by_horizon.png

  SmallMed (10 features):
  Feature                 |SHAP|      Bar
  ---------------------------------------------
  Mine_age                0.1692      ████████████████████
  Horizonte_feat          0.0896      ██████████
  Prod_Lag1               0.0794      █████████
  Mine_share              0.0785      █████████
  Prod_pct_change         0.0752      ████████
  Tendencia_5y            0.0697      ████████
  Company_Size            0.0336      ███
  Mine_Size               0.0161      █
  Is_Pandemic_Target      0.0139      █
  Cu_regime               0.0041      █

  LargeColossal (11 features):
  Feature                 |SHAP|      Bar
  ---------------------------------------------
  Prod_Lag1               0.0732      ████████████████████
  Mine_share              0.0543      ██████████████
  Mine_age                0.0442      ████████████
  Prod_pct_change         0.0396      ██████████
  Mine_Size               0.0303      ███

## Sección 9b — Resumen Comparativo de Modelos y Análisis por Mina

Compara todos los modelos evaluados (WR, Hfoc, MASE, MdAPE, Skill) y profundiza en el desempeño individual por mina:
- **Tabla resumen**: todos los experimentos ordenados por Hfoc% dentro de cada segmento
- **WR por horizonte**: top-5 vs bottom-5 minas — qué patrones explican la diferencia
- **Caso de estudio**: la mejor mina analizada en detalle (pred vs real, WR por horizonte, consistencia entre orígenes)


In [17]:

# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 9b — Resumen comparativo de modelos + Análisis por mina
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings; warnings.filterwarnings('ignore')

# ── Load best predictions CSV ─────────────────────────────────────────────────
_pred_path = 'outputs_best/all_predictions_annual_v8.csv'
try:
    _df = pd.read_csv(_pred_path)
except FileNotFoundError:
    _df = pd.read_csv('outputs_best/predicciones_anuales_baseline_extendido.csv')

_df['Mine'] = _df['Mine'].str.lower().str.strip()
H_FOCUS = [5, 6, 7]

def _stats(df, exp, hf=H_FOCUS):
    s = df[df['Exp']==exp].copy()
    if len(s)==0: return None
    s = s[(s['Actual']>0)&(s['Pred']>0)].copy()
    s['ae']  = abs(s['Actual']-s['Pred'])
    s['aen'] = abs(s['Actual']-s['Naive_Pred'])
    s['ape'] = s['ae']/s['Actual']*100
    s['b']   = (s['ae']<s['aen']).astype(int)
    wr   = s['b'].mean()*100
    foc  = s[s['Horizonte'].isin(hf)]['b'].mean()*100
    mase = s['ae'].mean()/s['aen'].mean()
    sk   = (1-mase)*100
    m50  = s.groupby('Mine')['b'].mean().ge(0.5).sum(); n=s['Mine'].nunique()
    sizes= s['Mine_Size'].dropna().unique()
    seg  = ('SM' if max(sizes)<=1 else ('LC' if min(sizes)>=2 else 'All')) if len(sizes)>0 else '?'
    return {'Modelo':exp, 'WR%':round(wr,1), 'Hfoc%':round(foc,1),
            'MASE':round(mase,3), 'MdAPE%':round(s['ape'].median(),1),
            'Skill%':round(sk,1), 'Minas≥50':f"{m50}/{n}", 'Seg':seg}

EXPS = [
    'LGB_LogRatio','LGB_MultiH','RF_MultiH','Ens_3070','Ens_Super_Annual',
    'LGB_LargeCol','XGB_LargeCol','RF_MultiH_LC','Ens_LC_3070',
    'Ens_Segmentado','SARIMAX',
]
rows = [r for r in (_stats(_df,e) for e in EXPS) if r]
_sb = pd.DataFrame(rows)

print("="*90)
print("RESUMEN COMPARATIVO — MODELOS ANUALES  (H_focus = H+5, H+6, H+7)")
print("="*90)
# Sort: SmallMed first, then LC, then All — within each segment by Hfoc%
_sb['_ord'] = _sb['Seg'].map({'SM':0,'LC':1,'All':2}).fillna(3)
_sb = _sb.sort_values(['_ord','Hfoc%'], ascending=[True,False]).drop(columns='_ord')
print(_sb[['Modelo','Seg','WR%','Hfoc%','MASE','MdAPE%','Skill%','Minas≥50']].to_string(index=False))

best_sm  = _sb[_sb['Seg']=='SM'].iloc[0]['Modelo'] if len(_sb[_sb['Seg']=='SM'])>0 else 'Ens_Super_Annual'
best_lc  = _sb[_sb['Seg']=='LC'].iloc[0]['Modelo'] if len(_sb[_sb['Seg']=='LC'])>0 else 'Ens_LC_3070'
best_all = _sb[_sb['Seg']=='All'].iloc[0]['Modelo'] if len(_sb[_sb['Seg']=='All'])>0 else 'Ens_Segmentado'
print(f"\n✅ Mejor SmallMed: {best_sm}  |  Mejor LargeCol: {best_lc}  |  Mejor global: {best_all}")

# ── Per-mine ranking table ────────────────────────────────────────────────────
_ens = _df[(_df['Exp']=='Ens_Segmentado')&(_df['Actual']>0)&(_df['Pred']>0)].copy()
_ens['ae']  = abs(_ens['Actual']-_ens['Pred'])
_ens['aen'] = abs(_ens['Actual']-_ens['Naive_Pred'])
_ens['ape'] = _ens['ae']/_ens['Actual']*100
_ens['b']   = (_ens['ae']<_ens['aen']).astype(int)

_mine_stats = _ens.groupby('Mine').apply(lambda g: pd.Series({
    'WR%': round(g['b'].mean()*100,1),
    'Hfoc%': round(g[g['Horizonte'].isin(H_FOCUS)]['b'].mean()*100,1),
    'MASE': round(g['ae'].mean()/g['aen'].mean(),3),
    'MdAPE%': round(g['ape'].median(),1),
    'n': len(g),
})).sort_values('WR%', ascending=False).reset_index()

# Add size label
try:
    _sbcsv = pd.read_csv('outputs_best/scoreboard_annual_v7.csv')
    _sbcsv['Mine'] = _sbcsv['Mine'].str.lower().str.strip()
    _size_map = _sbcsv.set_index('Mine')['Size_Label'].to_dict()
    _mine_stats['Size'] = _mine_stats['Mine'].map(_size_map).fillna('?')
except: _mine_stats['Size'] = '?'

print("\n" + "="*70)
print("RANKING POR MINA — Ens_Segmentado (H+1..H+7, todos los orígenes)")
print("="*70)
print(_mine_stats[['Mine','Size','WR%','Hfoc%','MASE','MdAPE%','n']].to_string(index=False))

# ── Fig 1: WR by horizon — top 5 vs bottom 5 ──────────────────────────────────
top5 = _mine_stats.head(5)['Mine'].tolist()
bot5 = _mine_stats.tail(5)['Mine'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Win Rate por Horizonte — Ens_Segmentado Anual", fontsize=13, fontweight='bold',
             color='white')
fig.patch.set_facecolor('#1e293b')
for ax in axes: ax.set_facecolor('#0f172a')

c_top = ['#22c55e','#4ade80','#34d399','#86efac','#6ee7b7']
c_bot = ['#ef4444','#f87171','#f97316','#fb923c','#fbbf24']

for i, (mines, colors, title, ax) in enumerate([
    (top5,  c_top, 'Top 5 Minas (mejores)',           axes[0]),
    (bot5,  c_bot, 'Bottom 5 Minas (más difíciles)',  axes[1]),
]):
    for j, mine in enumerate(mines):
        sg = _ens[_ens['Mine']==mine]
        wrh = sg.groupby('Horizonte')['b'].mean()*100
        ax.plot(wrh.index, wrh.values, 'o-', label=mine.title(), color=colors[j], linewidth=2, markersize=5)
    ax.axhline(50, color='#94a3b8', linestyle='--', alpha=0.6, linewidth=1.2, label='50% (naive)')
    ax.set_xlabel('Horizonte (años)', color='#94a3b8'); ax.set_ylabel('Win Rate %', color='#94a3b8')
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xticks(range(1,8))
    ax.tick_params(colors='#94a3b8'); ax.grid(alpha=0.2)
    [sp.set_color('#334155') for sp in ax.spines.values()]
    ax.legend(fontsize=8.5)

plt.tight_layout()
plt.savefig('outputs_best/wr_by_horizon_mines_annual.png', dpi=150, bbox_inches='tight',
            facecolor='#1e293b')
plt.show()
print("Saved → outputs_best/wr_by_horizon_mines_annual.png")

# ── Fig 2: Best mine case study ───────────────────────────────────────────────
best_mine = _mine_stats.iloc[0]['Mine']
bm_wr     = _mine_stats.iloc[0]['WR%']
_bm = _ens[_ens['Mine']==best_mine].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Caso de Estudio: {best_mine.title()}  (WR={bm_wr}%, MASE={_mine_stats.iloc[0]['MASE']})",
             fontsize=13, fontweight='bold', color='white')
fig.patch.set_facecolor('#1e293b')
for ax in axes: ax.set_facecolor('#0f172a')

# Scatter: predicted vs actual (colour = horizon)
sc = axes[0].scatter(_bm['Actual'], _bm['Pred'], c=_bm['Horizonte'],
                     cmap='RdYlGn_r', alpha=0.75, s=35, edgecolors='none')
_mn = min(_bm['Actual'].min(), _bm['Pred'].min())*0.88
_mx = max(_bm['Actual'].max(), _bm['Pred'].max())*1.08
axes[0].plot([_mn,_mx],[_mn,_mx], '--', color='#94a3b8', linewidth=1.2)
plt.colorbar(sc, ax=axes[0], label='Horizonte')
axes[0].set_xlabel('Real (kt)', color='#94a3b8'); axes[0].set_ylabel('Predicción (kt)', color='#94a3b8')
axes[0].set_title('Pred vs Real', color='white')
axes[0].tick_params(colors='#94a3b8'); axes[0].grid(alpha=0.2)

# WR by horizon bar
_wrh = _bm.groupby('Horizonte')['b'].mean()*100
_cols = ['#22c55e' if v>=50 else '#ef4444' for v in _wrh.values]
axes[1].bar(_wrh.index, _wrh.values, color=_cols, alpha=0.85, edgecolor='#1e293b')
axes[1].axhline(50, color='#f59e0b', linestyle='--', linewidth=1.2)
axes[1].set_xlabel('Horizonte (años)', color='#94a3b8'); axes[1].set_ylabel('Win Rate %', color='#94a3b8')
axes[1].set_title('Win Rate por Horizonte', color='white')
axes[1].set_xticks(range(1,8)); axes[1].grid(axis='y', alpha=0.2)
axes[1].tick_params(colors='#94a3b8')

# Per-origin WR
_per_orig = _bm.groupby('Origin')['b'].mean()*100
axes[2].bar(_per_orig.index.astype(str), _per_orig.values,
            color=['#22c55e' if v>=50 else '#ef4444' for v in _per_orig.values],
            alpha=0.85, edgecolor='#1e293b')
axes[2].axhline(50, color='#f59e0b', linestyle='--', linewidth=1.2)
axes[2].set_xlabel('Origen', color='#94a3b8'); axes[2].set_ylabel('Win Rate %', color='#94a3b8')
axes[2].set_title('Win Rate por Origen', color='white')
axes[2].tick_params(axis='x', rotation=45, colors='#94a3b8'); axes[2].tick_params(axis='y', colors='#94a3b8')
axes[2].grid(axis='y', alpha=0.2)
[sp.set_color('#334155') for ax in axes for sp in ax.spines.values()]

plt.tight_layout()
plt.savefig(f"outputs_best/case_study_{best_mine.replace(' ','_')}_annual.png",
            dpi=150, bbox_inches='tight', facecolor='#1e293b')
plt.show()
print(f"Saved → outputs_best/case_study_{best_mine.replace(' ','_')}_annual.png")

# ── Per-origin table for best mine ────────────────────────────────────────────
print(f"\nDetalle por origen — {best_mine.title()}")
_per = _bm.groupby('Origin').apply(lambda g: pd.Series({
    'WR%': round(g['b'].mean()*100,1),
    'MAE':  round(g['ae'].mean(),1),
    'MASE': round(g['ae'].mean()/g['aen'].mean(),3),
    'MdAPE%': round(g['ape'].median(),1),
    'n': len(g),
})).reset_index()
print(_per.to_string(index=False))


RESUMEN COMPARATIVO — MODELOS ANUALES  (H_focus = H+5, H+6, H+7)
          Modelo Seg  WR%  Hfoc%  MASE  MdAPE%  Skill% Minas≥50
       RF_MultiH  SM 54.2   61.0 0.879    21.5    12.1     6/10
        Ens_3070  SM 58.1   61.0 0.967    18.9     3.3     6/10
Ens_Super_Annual  SM 58.1   61.0 0.968    19.2     3.2     6/10
      LGB_MultiH  SM 52.0   57.6 0.902    21.5     9.8     7/10
    LGB_LogRatio  SM 49.1   47.5 1.003    25.1    -0.3     4/10
    RF_MultiH_LC  LC 49.7   49.3 1.138    13.9   -13.8    12/21
     Ens_LC_3070  LC 50.4   48.5 1.097    14.3    -9.7    10/21
    LGB_LargeCol  LC 47.2   45.6 1.123    14.7   -12.3    10/21
    XGB_LargeCol  LC 45.2   44.1 1.128    14.8   -12.8     8/21
  Ens_Segmentado All 53.1   55.2 1.079    14.9    -7.9    14/24
         SARIMAX All 48.7   51.7 1.120    14.7   -12.0    10/22

✅ Mejor SmallMed: RF_MultiH  |  Mejor LargeCol: RF_MultiH_LC  |  Mejor global: Ens_Segmentado

RANKING POR MINA — Ens_Segmentado (H+1..H+7, todos los orígenes)
      

Saved → outputs_best/wr_by_horizon_mines_annual.png


Saved → outputs_best/case_study_ministro_hales_annual.png

Detalle por origen — Ministro Hales
 Origin   WR%   MAE  MASE  MdAPE%   n
   2013 100.0 147.8 0.928    77.3 7.0
   2014 100.0  49.9 0.871    23.9 7.0
   2015  71.4  47.5 0.912    26.5 7.0
   2016 100.0  60.9 0.915    35.4 7.0
   2017  42.9  57.5 0.992    40.2 7.0
   2018 100.0  40.6 0.914    26.3 7.0


In [18]:
import subprocess, warnings
warnings.filterwarnings('ignore')
try:
    import shap
except ImportError:
    subprocess.run(['pip','install','--quiet','shap'], check=True); import shap
import matplotlib.pyplot as plt
import numpy as np

# ─── Thesis style ─────────────────────────────────────────────────────────────
_THESIS = {
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#e5e7eb', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#374151', 'axes.labelcolor': '#111827',
    'xtick.color': '#374151', 'ytick.color': '#374151',
    'text.color': '#111827', 'font.family': 'DejaVu Sans',
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'legend.fontsize': 9, 'savefig.facecolor': 'white',
    'savefig.bbox': 'tight',
}
plt.rcParams.update(_THESIS)

# ── SHAP: usa fm_sm / fm_lc entrenados en Sección 8 ──────────────────────────
_Xsm_a = pd.DataFrame(X_sm_ann, columns=E6_MULTI_SM)
_Xlc_a = pd.DataFrame(X_lc_ann, columns=E6_MULTI_LC)
np.random.seed(42)
_Xsm_s = _Xsm_a.sample(min(600, len(_Xsm_a)), random_state=42)
_Xlc_s = _Xlc_a.sample(min(400, len(_Xlc_a)), random_state=42)

print("Computing SHAP — SmallMed (fm_sm) …")
_exp_sm = shap.TreeExplainer(fm_sm)
_sv_sm  = _exp_sm.shap_values(_Xsm_s)
print("Computing SHAP — LargeColossal (fm_lc) …")
_exp_lc = shap.TreeExplainer(fm_lc)
_sv_lc  = _exp_lc.shap_values(_Xlc_s)

# ── Plot 1: Importancia global (barra) ────────────────────────────────────────
_THESIS_SM = '#4e79a7'
_THESIS_LC = '#9467bd'

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("SHAP — Importancia Global de Features — Modelo Anual",
             fontsize=13, fontweight='bold')

plt.sca(axes[0])
shap.summary_plot(_sv_sm, _Xsm_s, feature_names=E6_MULTI_SM, plot_type='bar',
                  show=False, max_display=len(E6_MULTI), color=_THESIS_SM)
axes[0].set_title("Segmento SmallMed (H+1–H+7)", fontweight='bold')
axes[0].set_facecolor('white')

plt.sca(axes[1])
shap.summary_plot(_sv_lc, _Xlc_s, feature_names=E6_MULTI_LC, plot_type='bar',
                  show=False, max_display=len(E6_MULTI), color=_THESIS_LC)
axes[1].set_title("Segmento LargeColossal (H+1–H+7)", fontweight='bold')
axes[1].set_facecolor('white')

plt.tight_layout()
plt.savefig(f'{EXPORT_DIR}/shap_annual_importance.png', dpi=200)
plt.show(); print("Saved → shap_annual_importance.png")

# ── Plot 2: Beeswarm SmallMed ─────────────────────────────────────────────────
fig = plt.figure(figsize=(10, 6))
plt.title("SmallMed — Dirección de Efectos SHAP (beeswarm)", fontsize=12, fontweight='bold')
shap.summary_plot(_sv_sm, _Xsm_s, feature_names=E6_MULTI_SM,
                  show=False, max_display=len(E6_MULTI))
ax = plt.gca(); ax.set_facecolor('white')
plt.tight_layout()
plt.savefig(f'{EXPORT_DIR}/shap_annual_beeswarm_sm.png', dpi=200)
plt.show(); print("Saved → shap_annual_beeswarm_sm.png")

# ── Plot 3: SHAP por horizonte (buckets H+1-2, H+3-4, H+5-7) ─────────────────
_BUCKET_COLORS = ['#1d4ed8', '#0d9488', '#ea580c']
_buckets_a = {'H+1–2': (1, 2), 'H+3–4': (3, 4), 'H+5–7': (5, 7)}
_bshap_a = {}
for bname, (lo, hi) in _buckets_a.items():
    mask = (_Xsm_s['Horizonte_feat'] >= lo) & (_Xsm_s['Horizonte_feat'] <= hi)
    if mask.sum() >= 5:
        _bshap_a[bname] = np.abs(_sv_sm[mask.values]).mean(axis=0)

if _bshap_a:
    fig, ax = plt.subplots(figsize=(12, 4))
    x = np.arange(len(E6_MULTI)); w = 0.25
    for i, (bname, vals) in enumerate(_bshap_a.items()):
        ax.bar(x + i*w, vals, w, label=bname,
               color=_BUCKET_COLORS[i], alpha=0.85, edgecolor='white')
    ax.set_xticks(x + w)
    ax.set_xticklabels(E6_MULTI_SM, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('|SHAP| medio')
    ax.set_title('SmallMed — Importancia SHAP por Horizonte Anual', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{EXPORT_DIR}/shap_annual_by_horizon.png', dpi=200)
    plt.show(); print("Saved → shap_annual_by_horizon.png")

# ── Tabla resumen ─────────────────────────────────────────────────────────────
_ms = np.abs(_sv_sm).mean(axis=0)
_ml = np.abs(_sv_lc).mean(axis=0)
print(f"\n  SmallMed features ({len(E6_MULTI_SM)}):")
print(f"  {'Feature':22s}  {'|SHAP|':10s}  Barra")
print("  " + "-"*45)
for i in np.argsort(_ms)[::-1]:
    print(f"  {E6_MULTI_SM[i]:22s}  {_ms[i]:.4f}  "
          f"{'█' * max(1, int(_ms[i] / _ms.max() * 20))}")
print(f"\n  LargeColossal features ({len(E6_MULTI_LC)}):")
print(f"  {'Feature':22s}  {'|SHAP|':10s}  Barra")
print("  " + "-"*45)
for i in np.argsort(_ml)[::-1]:
    print(f"  {E6_MULTI_LC[i]:22s}  {_ml[i]:.4f}  "
          f"{'█' * max(1, int(_ml[i] / _ml.max() * 20))}")

Computing SHAP — SmallMed (fm_sm) …


Computing SHAP — LargeColossal (fm_lc) …


Saved → shap_annual_importance.png
Saved → shap_annual_beeswarm_sm.png


Saved → shap_annual_by_horizon.png

  SmallMed features (10):
  Feature                 |SHAP|      Barra
  ---------------------------------------------
  Mine_age                0.1692  ████████████████████
  Horizonte_feat          0.0896  ██████████
  Prod_Lag1               0.0794  █████████
  Mine_share              0.0785  █████████
  Prod_pct_change         0.0752  ████████
  Tendencia_5y            0.0697  ████████
  Company_Size            0.0336  ███
  Mine_Size               0.0161  █
  Is_Pandemic_Target      0.0139  █
  Cu_regime               0.0041  █

  LargeColossal features (11):
  Feature                 |SHAP|      Barra
  ---------------------------------------------
  Prod_Lag1               0.0732  ████████████████████
  Mine_share              0.0543  ██████████████
  Mine_age                0.0442  ████████████
  Prod_pct_change         0.0396  ██████████
  Mine_Size               0.0303  ████████
  Capital_Stock_Lag1      0.0303  ████████
  Horizonte_feat    

In [19]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

# ─── Thesis style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#e5e7eb', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#374151', 'xtick.color': '#374151', 'ytick.color': '#374151',
    'text.color': '#111827', 'font.size': 10, 'savefig.facecolor': 'white',
    'savefig.bbox': 'tight',
})
_BLUE = '#1d4ed8'; _NAVY = '#1e3a5f'; _GRAY = '#6b7280'

# ── Proyecciones 2026-2032 con bandas CI ──────────────────────────────────────
_df_proj  = pd.read_csv(f'{EXPORT_DIR}/proyecciones_minas_2026_2032.csv')
_hist_ann = df_raw[df_raw['Anio'] >= 2010][['Match_Key', 'Anio', 'Produccion']].copy()
_hist_ann.columns = ['Mine', 'Year', 'Production']

top_mines_ann = mine_agg.nlargest(12, 'WR')['Mine'].tolist()

fig, axes = plt.subplots(3, 4, figsize=(18, 11))
fig.suptitle('Proyecciones Anuales 2026–2032 — Ensamble Segmentado\n(base ± IC q10–q90, top 12 por WR)',
             fontsize=13, fontweight='bold')

for idx, mine in enumerate(top_mines_ann[:12]):
    ax = axes[idx // 4][idx % 4]
    h_m = _hist_ann[_hist_ann['Mine'] == mine].sort_values('Year')
    p_m = _df_proj[_df_proj['Mine'] == mine].sort_values('ForecastYear')
    if not h_m.empty:
        ax.plot(h_m['Year'], h_m['Production'],
                'o-', color=_NAVY, lw=1.8, ms=4, label='Histórico')
    if not p_m.empty:
        origin_y    = 2025
        origin_prod = p_m['Origin_Prod'].iloc[0]
        years  = [origin_y] + p_m['ForecastYear'].tolist()
        preds  = [origin_prod] + p_m['Pred'].tolist()
        lowers = [origin_prod] + p_m['Lower'].tolist()
        uppers = [origin_prod] + p_m['Upper'].tolist()
        ax.plot(years, preds, 'o--', color=_THESIS_SM, lw=2, ms=4, label='Pronóstico')
        ax.fill_between(years, lowers, uppers, color=_THESIS_SM, alpha=0.15, label='IC 80%')
        ax.axhline(p_m['Naive_Pred'].iloc[0], color=_GRAY, lw=1.2, ls=':', label='Naïve')
    wr_v = mine_agg[mine_agg['Mine'] == mine]['WR'].values
    sk_v = mine_agg[mine_agg['Mine'] == mine]['Skill'].values
    mine_disp = (mine.replace('centinela_centinela_sulfuros_', 'Centinela Sulf.')
                     .replace('centinela_centinela_óxidos_', 'Centinela Óx.')
                     .replace('capstone copper (4)', 'Capstone (4)')
                     .replace('_', ' ').title())
    lbl = (f'{mine_disp}\nWR={wr_v[0]*100:.0f}%  Skill={sk_v[0]:+.0f}%'
           if len(wr_v) else mine_disp)
    ax.set_title(lbl, fontsize=8.5, fontweight='bold')
    ax.set_ylabel('kt Cu', fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))
    ax.tick_params(axis='both', labelsize=7)
    ax.tick_params(axis='x', rotation=30)
    if idx == 0:
        ax.legend(fontsize=7)

# Global legend
handles = [
    plt.Line2D([0],[0], color=_NAVY, marker='o', ls='-',  ms=4, lw=1.8, label='Histórico'),
    plt.Line2D([0],[0], color=_THESIS_SM, marker='o', ls='--', ms=4, lw=2,   label='Pronóstico'),
    Patch(facecolor=_THESIS_SM, alpha=0.2, label='IC 80 %'),
    plt.Line2D([0],[0], color=_GRAY, ls=':', lw=1.2, label='Naïve'),
]
fig.legend(handles=handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.02), fontsize=9)
plt.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.savefig(f'{EXPORT_DIR}/projections_plot_annual.png', dpi=200)
plt.show()
print(f"Saved → {EXPORT_DIR}/projections_plot_annual.png")

Saved → outputs_best/projections_plot_annual.png


## Sección 10 — Proyecciones v8c (v7 + Capital_Stock_Lag1)

Re-entrena con todos los datos disponibles (hasta 2025) usando los parámetros Optuna
validados en v7, pero con el feature set v8c (10 features: v7 + Capital_Stock_Lag1).

**Cambios respecto a v7:**
- `Capital_Stock_Lag1` añadido → mejora H+5-7 WR +2pp sin dañar H+1
- `Company_Size` restaurado (faltaba en v8b)
- `Mine_age` cappado a 36 (máximo del training) → evita OOD en proyecciones
- Eliminados: `Prod_vs_HistMax`, `Is_Decline` (dañaban H+1 -3pp), `SEIA_pipeline_log` (sin datos)

Outputs: `proyecciones_minas_2026_2032_v8c.csv` (base) + `proyecciones_escenarios_2026_2032_v8c.csv`

In [20]:
import math, json

ORIGIN_YEAR_V8C  = BASE_YEAR
HORIZONS_V8C     = HORIZONS
CU_SCENARIOS_V8C = {"bear": 0.2, "base": 0.5, "bull": 0.8}
EXPORT_V8C       = EXPORT_DIR

ms_v8c = compute_mine_size(df_raw, ORIGIN_YEAR_V8C)
df_fp8c = df_feats.copy()
df_fp8c['Mine_Size'] = df_fp8c['Match_Key'].map(ms_v8c).fillna(1).astype(int)

assert 'Company_Size' in E6_BASE_SM and 'Capital_Stock_Lag1' in E6_BASE_LC
assert 'SEIA_pipeline_log' not in E6_BASE_SM
print(f"v8c SM features ({len(E6_MULTI_SM)}): {E6_MULTI_SM}")
print(f"v8c LC features ({len(E6_MULTI_LC)}): {E6_MULTI_LC}")

def _build_v8c_train(size_filt, feats):
    frames = []
    for h_tr in HORIZONS_V8C:
        df_ht = df_fp8c.copy()
        df_ht['Target']             = df_ht.groupby('Match_Key')['Produccion'].shift(-h_tr)
        df_ht['Target_Year']        = df_ht['Anio'] + h_tr
        df_ht['Horizonte_feat']     = h_tr
        df_ht['Is_Pandemic_Target'] = df_ht['Target_Year'].isin(PANDEMIC_YEARS).astype(int)
        sub = df_ht[
            (df_ht['Anio']        <= ORIGIN_YEAR_V8C) &
            (df_ht['Target_Year'] <= ORIGIN_YEAR_V8C) &
            (~df_ht['Match_Key'].isin(EXCLUDE_MINES)) &
            (df_ht['Mine_Size'].isin(size_filt)) &
            (df_ht['Prod_Lag1']   > 0)
        ].dropna(subset=feats + ['Target', 'Produccion'])
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

print("\nBuilding v8c training data...")
train_sm_v8c = _build_v8c_train([0, 1], E6_MULTI_SM)
train_lc_v8c = _build_v8c_train([2, 3], E6_MULTI_LC)
print(f"  SmallMed : {len(train_sm_v8c):,} rows | LargeColo: {len(train_lc_v8c):,} rows")

y_sm_v8c = np.clip(np.log((train_sm_v8c['Target']+1e-6)/(train_sm_v8c['Produccion']+1e-6)).values, -3, 3)
y_lc_v8c = np.clip(np.log((train_lc_v8c['Target']+1e-6)/(train_lc_v8c['Produccion']+1e-6)).values, -3, 3)
X_sm_v8c = train_sm_v8c[E6_MULTI_SM].fillna(0).values
X_lc_v8c = train_lc_v8c[E6_MULTI_LC].fillna(0).values

with open(f'{EXPORT_V8C}/optuna_params_annual.json') as f:
    _op = json.load(f)

def _ann_trio(X, y, params, name):
    base = {k:v for k,v in params.items() if k not in ('objective','verbose','n_estimators','num_boost_round')}
    mdl     = lgb.LGBMRegressor(**base, objective='regression',  verbose=-1, n_estimators=500)
    mdl_q10 = lgb.LGBMRegressor(**base, objective='quantile', alpha=0.10, verbose=-1, n_estimators=500)
    mdl_q90 = lgb.LGBMRegressor(**base, objective='quantile', alpha=0.90, verbose=-1, n_estimators=500)
    mdl.fit(X, y); mdl_q10.fit(X, y); mdl_q90.fit(X, y)
    print(f"  {name}: trained ({X.shape[0]:,} rows, {X.shape[1]} feats)")
    return mdl, mdl_q10, mdl_q90

print("\nTraining v8c models...")
mdl_sm_v8c, mdl_sm_q10, mdl_sm_q90 = _ann_trio(X_sm_v8c, y_sm_v8c, _op.get('LGB_MultiH',{}),    'SmallMed LGB')
mdl_lc_v8c, mdl_lc_q10, mdl_lc_q90 = _ann_trio(X_lc_v8c, y_lc_v8c, _op.get('LGB_MultiH_LC',{}), 'LargeCol LGB')

# RF component for SM blend (no quantile, CI from LGB quantile models)
def _rf_mean(X, y, params, name):
    from sklearn.ensemble import RandomForestRegressor
    _rf_keys = {'n_estimators','max_depth','min_samples_leaf','max_features'}
    base = {k:v for k,v in params.items() if k in _rf_keys}
    base.setdefault('n_estimators', 200); base.setdefault('max_depth', 8)
    base.setdefault('min_samples_leaf', 5); base.setdefault('max_features', 'sqrt')
    m = RandomForestRegressor(**base, random_state=42, n_jobs=-1).fit(X, y)
    print(f"  {name}: RF mean trained ({X.shape[0]:,} rows, {X.shape[1]} feats)")
    return m

p_rf_sm = _op.get('RF_MultiH', {})
mdl_rf_sm_v8c = _rf_mean(X_sm_v8c, y_sm_v8c, p_rf_sm, "SmallMed RF")

# Bias correction removed: LOO analysis showed H+1 hurt (-16pp) while H+7 only +2pp
# Projections use raw model predictions (no bias adjustment)

# ── Generate projections 2026-2032 ────────────────────────────────────────────
SIZE_LABEL_MAP = {0:'Small',1:'Medium',2:'Large',3:'Colossal'}
rows_v8c = []
rows_scen_v8c = []

origin_data = df_feats[df_feats['Anio'] == ORIGIN_YEAR_V8C].copy()
origin_data['Mine_Size'] = origin_data['Match_Key'].map(ms_v8c).fillna(1).astype(int)

for _, row in origin_data.iterrows():
    mk = row['Match_Key']
    if mk in EXCLUDE_MINES: continue
    seg = row['Mine_Size']
    mdl, mdl_q10, mdl_q90 = (mdl_sm_v8c, mdl_sm_q10, mdl_sm_q90) if seg <= 1 \
                           else (mdl_lc_v8c, mdl_lc_q10, mdl_lc_q90)
    orig_prod = row['Produccion']
    if orig_prod <= 0: continue

    for h in HORIZONS_V8C:
        feat_row = row.copy()
        feat_row['Is_Pandemic_Target'] = 0
        feat_row['Horizonte_feat']     = h
        _feats_proj = E6_MULTI_SM if seg <= 1 else E6_MULTI_LC
        X_proj = feat_row[_feats_proj].fillna(0).values.reshape(1, -1)

        if seg <= 1:
            # SM: blend 50% LGB_MultiH + 50% RF_MultiH
            lr = 0.5 * float(mdl.predict(X_proj)[0]) + 0.5 * float(mdl_rf_sm_v8c.predict(X_proj)[0])
        else:
            lr = float(mdl.predict(X_proj)[0])
        lr_q10  = float(mdl_q10.predict(X_proj)[0])
        lr_q90  = float(mdl_q90.predict(X_proj)[0])
        lr_lo   = min(lr_q10, lr_q90)
        lr_hi   = max(lr_q10, lr_q90)

        pred  = np.exp(lr)    * orig_prod
        lower = np.exp(lr_lo) * orig_prod
        upper = np.exp(lr_hi) * orig_prod

        rows_v8c.append({
            'Mine': mk, 'ForecastYear': ORIGIN_YEAR_V8C + h,
            'Horizonte': h, 'Pred': round(pred, 1),
            'Naive_Pred': round(orig_prod, 1),
            'Lower': round(lower, 1), 'Upper': round(upper, 1),
            'Origin_Prod': round(orig_prod, 1),
            'Mine_Size': seg, 'Size_Label': SIZE_LABEL_MAP.get(seg, '?'),
            'Company_Size': int(row['Company_Size']),
            'Segment': 'SM' if seg <= 1 else 'LC',
        })

        for scen_name, cu_val in CU_SCENARIOS_V8C.items():
            feat_row_s = feat_row.copy()
            feat_row_s['Cu_regime'] = cu_val
            X_s = feat_row_s[_feats_proj].fillna(0).values.reshape(1, -1)
            if seg <= 1:
                lr_s = 0.5 * float(mdl.predict(X_s)[0]) + 0.5 * float(mdl_rf_sm_v8c.predict(X_s)[0])
            else:
                lr_s = float(mdl.predict(X_s)[0])
            rows_scen_v8c.append({
                'Mine': mk, 'ForecastYear': ORIGIN_YEAR_V8C + h,
                'Horizonte': h, 'Scenario': scen_name,
                'Pred': round(np.exp(lr_s) * orig_prod, 1),
                'Origin_Prod': round(orig_prod, 1),
                'Mine_Size': seg, 'Size_Label': SIZE_LABEL_MAP.get(seg, '?'),
                'Segment': 'SM' if seg <= 1 else 'LC',
            })

df_proj_v8c      = pd.DataFrame(rows_v8c)
df_proj_scen_v8c = pd.DataFrame(rows_scen_v8c)

out_base = f'{EXPORT_V8C}/proyecciones_minas_2026_2032_v8c.csv'
out_scen = f'{EXPORT_V8C}/proyecciones_escenarios_2026_2032_v8c.csv'
df_proj_v8c.to_csv(out_base, index=False)
df_proj_scen_v8c.to_csv(out_scen, index=False)
print(f"\nSaved: {out_base}  ({len(df_proj_v8c)} rows)")
print(f"Saved: {out_scen}  ({len(df_proj_scen_v8c)} rows)")

# ── Summary ───────────────────────────────────────────────────────────────────
chile = df_proj_v8c.groupby('ForecastYear')['Pred'].sum().reset_index()
print(f"\nChile total v8c (bias-corrected, base scenario):")
for _, r in chile.iterrows():
    print(f"  {int(r['ForecastYear'])}: {r['Pred']:,.0f} kt")

print(f"\n  SM segment uses: 50% LGB_MultiH + 50% RF_MultiH (no bias correction)")

v8c SM features (10): ['Company_Size', 'Mine_Size', 'Prod_Lag1', 'Tendencia_5y', 'Prod_pct_change', 'Mine_age', 'Mine_share', 'Is_Pandemic_Target', 'Cu_regime', 'Horizonte_feat']
v8c LC features (11): ['Company_Size', 'Mine_Size', 'Prod_Lag1', 'Tendencia_5y', 'Prod_pct_change', 'Mine_age', 'Mine_share', 'Is_Pandemic_Target', 'Cu_regime', 'Capital_Stock_Lag1', 'Horizonte_feat']

Building v8c training data...
  SmallMed : 1,229 rows | LargeColo: 2,758 rows

Training v8c models...


  SmallMed LGB: trained (1,229 rows, 10 feats)


  LargeCol LGB: trained (2,758 rows, 11 feats)
  SmallMed RF: RF mean trained (1,229 rows, 10 feats)



Saved: outputs_best/proyecciones_minas_2026_2032_v8c.csv  (217 rows)
Saved: outputs_best/proyecciones_escenarios_2026_2032_v8c.csv  (651 rows)

Chile total v8c (bias-corrected, base scenario):
  2026: 5,191 kt
  2027: 4,941 kt
  2028: 4,767 kt
  2029: 4,725 kt
  2030: 4,623 kt
  2031: 4,505 kt
  2032: 4,421 kt

  SM segment uses: 50% LGB_MultiH + 50% RF_MultiH (no bias correction)
